In [ ]:
# FIRST CELL — persistent kernel-restart recovery
# Run this cell first after a kernel restart. It restores the namespace saved
# after the last successfully completed cell. The interrupted cell must then
# be rerun from its beginning.

import os as _os
import sys as _sys
import json as _json
import pickle as _pickle
import hashlib as _hashlib
import importlib as _importlib
import importlib.util as _importlib_util
import subprocess as _subprocess
import types as _types
import traceback as _traceback
from pathlib import Path as _RecoveryPath
from datetime import datetime as _RecoveryDateTime

if _importlib_util.find_spec("cloudpickle") is None:
    _subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "-q", "cloudpickle"
    ])

import cloudpickle as _cloudpickle

_RECOVERY_SCHEMA = "updated_shohoj_pro_max_ultra_v1"
_RECOVERY_ROOT = (
    _RecoveryPath(_os.environ["TOX21_RESUME_DIR"]).expanduser()
    if _os.environ.get("TOX21_RESUME_DIR")
    else _RecoveryPath.cwd() / ".updated_shohoj_pro_max_ultra_resume"
)
_RECOVERY_ROOT.mkdir(parents=True, exist_ok=True)

_RECOVERY_STATE_FILE = _RECOVERY_ROOT / "namespace_state.pkl"
_RECOVERY_META_FILE = _RECOVERY_ROOT / "resume_metadata.json"
_RECOVERY_LOG_FILE = _RECOVERY_ROOT / "resume_errors.log"

if _os.environ.get("TOX21_CLEAR_RESUME", "0") == "1":
    for _path in (_RECOVERY_STATE_FILE, _RECOVERY_META_FILE):
        try:
            _path.unlink(missing_ok=True)
        except Exception:
            pass

_RECOVERY_SKIP_NAMES = {
    "In", "Out", "get_ipython", "exit", "quit", "open",
    "cloudpickle",
}

def _recovery_write_log(message):
    try:
        with _RECOVERY_LOG_FILE.open("a", encoding="utf-8") as _handle:
            _handle.write(
                f"[{_RecoveryDateTime.now().isoformat(timespec='seconds')}] "
                f"{message}\n"
            )
    except Exception:
        pass

def _recovery_restore():
    if not _RECOVERY_STATE_FILE.exists():
        return None

    try:
        with _RECOVERY_STATE_FILE.open("rb") as _handle:
            _payload = _cloudpickle.load(_handle)

        if _payload.get("schema") != _RECOVERY_SCHEMA:
            return None

        for _alias, _module_name in _payload.get("modules", {}).items():
            try:
                globals()[_alias] = _importlib.import_module(_module_name)
            except Exception as _exc:
                _recovery_write_log(
                    f"Module restore skipped for {_alias}={_module_name}: {_exc}"
                )

        globals().update(_payload.get("state", {}))
        return _payload.get("metadata", {})

    except Exception:
        _recovery_write_log(
            "Namespace restore failed:\n" + _traceback.format_exc()
        )
        return None

_RECOVERED_METADATA = _recovery_restore()

def _recovery_collect_namespace():
    _ip = get_ipython()
    _namespace = _ip.user_ns if _ip is not None else globals()

    _state = {}
    _modules = {}

    for _name, _value in list(_namespace.items()):
        if (
            not _name
            or _name.startswith("_")
            or _name in _RECOVERY_SKIP_NAMES
        ):
            continue

        if isinstance(_value, _types.ModuleType):
            _modules[_name] = _value.__name__
            continue

        _state[_name] = _value

    return _state, _modules

def _recovery_atomic_snapshot(metadata):
    _state, _modules = _recovery_collect_namespace()
    _payload = {
        "schema": _RECOVERY_SCHEMA,
        "metadata": metadata,
        "modules": _modules,
        "state": _state,
    }

    _temporary = _RECOVERY_STATE_FILE.with_suffix(".tmp")

    try:
        with _temporary.open("wb") as _handle:
            _cloudpickle.dump(
                _payload,
                _handle,
                protocol=_pickle.HIGHEST_PROTOCOL,
            )
        _temporary.replace(_RECOVERY_STATE_FILE)

    except Exception:
        # Fallback: exclude the individual objects that cannot be serialized.
        _recovery_write_log(
            "Full namespace snapshot failed; using filtered fallback:\n"
            + _traceback.format_exc()
        )

        _filtered_state = {}
        for _name, _value in _state.items():
            try:
                _cloudpickle.dumps(
                    _value,
                    protocol=_pickle.HIGHEST_PROTOCOL,
                )
                _filtered_state[_name] = _value
            except Exception as _exc:
                _recovery_write_log(
                    f"Skipped non-serializable variable {_name}: {_exc}"
                )

        _payload["state"] = _filtered_state
        with _temporary.open("wb") as _handle:
            _cloudpickle.dump(
                _payload,
                _handle,
                protocol=_pickle.HIGHEST_PROTOCOL,
            )
        _temporary.replace(_RECOVERY_STATE_FILE)

    _RECOVERY_META_FILE.write_text(
        _json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

def _recovery_post_run_cell(result):
    _raw_cell = getattr(getattr(result, "info", None), "raw_cell", "") or ""
    _preview = next(
        (line.strip() for line in _raw_cell.splitlines() if line.strip()),
        "<empty cell>",
    )[:180]

    _failed = (
        getattr(result, "error_before_exec", None) is not None
        or getattr(result, "error_in_exec", None) is not None
    )

    if _failed:
        _failure_metadata = {
            "schema": _RECOVERY_SCHEMA,
            "status": "cell_failed",
            "failed_cell_preview": _preview,
            "failed_cell_sha256": _hashlib.sha256(
                _raw_cell.encode("utf-8")
            ).hexdigest(),
            "recorded_at": _RecoveryDateTime.now().isoformat(
                timespec="seconds"
            ),
        }
        try:
            _RECOVERY_META_FILE.write_text(
                _json.dumps(_failure_metadata, indent=2),
                encoding="utf-8",
            )
        except Exception:
            pass
        return

    _metadata = {
        "schema": _RECOVERY_SCHEMA,
        "status": "last_cell_completed",
        "execution_count": getattr(result, "execution_count", None),
        "last_cell_preview": _preview,
        "last_cell_sha256": _hashlib.sha256(
            _raw_cell.encode("utf-8")
        ).hexdigest(),
        "saved_at": _RecoveryDateTime.now().isoformat(
            timespec="seconds"
        ),
        "resume_instruction": (
            "After restart, run the first recovery cell, then rerun the "
            "interrupted cell from its beginning."
        ),
    }

    try:
        _recovery_atomic_snapshot(_metadata)
    except Exception:
        _recovery_write_log(
            "Snapshot callback failed:\n" + _traceback.format_exc()
        )

_ipython = get_ipython()
if _ipython is not None:
    _old_callback = getattr(
        _ipython,
        "_tox21_max_ultra_recovery_callback",
        None,
    )
    if _old_callback is not None:
        try:
            _ipython.events.unregister("post_run_cell", _old_callback)
        except Exception:
            pass

    _ipython.events.register("post_run_cell", _recovery_post_run_cell)
    _ipython._tox21_max_ultra_recovery_callback = _recovery_post_run_cell

if _RECOVERED_METADATA:
    print("Previous notebook state restored.")
    print(
        "Last completed cell:",
        _RECOVERED_METADATA.get("last_cell_preview", "unknown"),
    )
    print("Rerun the interrupted cell from its beginning, then continue.")
else:
    print("No compatible recovery snapshot was found. Starting a fresh run.")


# Updated_Shohoj_Pro_Max_Ultra — Tox21 Drug Toxicity Prediction

### Beginner-friendly, leakage-aware and strongly regularized ML/DL notebook

This notebook preserves the modeling, preprocessing, regularization and hybrid architecture of `Updated_shohoj_Pro.ipynb`. Only the requested evaluation/output and restart-recovery behavior were revised.

Main changes in this version:

- Evaluation output is restricted to **ROC-AUC** and **Accuracy**.
- PR-AUC, Balanced Accuracy, Sensitivity, Specificity and F1-score are not calculated, displayed, plotted or exported.
- Accuracy uses one consistent fixed threshold of **0.50** for every model and endpoint.
- The lower validation-selected thresholds used previously are removed. This is expected to increase ordinary Accuracy on the highly imbalanced Tox21 endpoints without changing ROC-AUC scores.
- Endpoint-by-endpoint messages such as `Training Regularized Random Forest | NR-AR` are hidden, while the endpoint models are still trained normally.
- Classical fitting diagnostics continue to use 3-fold out-of-fold predictions.
- The DenseNet121 + Chemical XGBoost feature-level hybrid, all model architectures and all regularization settings are otherwise preserved.
- The first code cell provides persistent kernel-restart recovery. It restores the namespace through the last successfully completed cell; the interrupted cell is then rerun from its beginning.

> Important: this notebook contains revised code, not fabricated results. Run the notebook to obtain measured ROC-AUC and Accuracy values.


## 0. Restart-safe execution

The first code cell is the recovery cell.

- On a fresh run, execute the notebook from top to bottom.
- After a kernel restart, execute **only the first recovery cell**. It restores the Python namespace saved after the last successfully completed cell.
- Then rerun the interrupted cell from its beginning and continue normally.
- A Python cell cannot resume from the exact interrupted line; recovery is at the cell boundary.
- Set `TOX21_CLEAR_RESUME=1` before running the first cell when a completely fresh run is required.


In [ ]:
import os
import sys
import subprocess
import importlib.util

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")

PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "rdkit": "rdkit",
    "iterstrat": "iterative-stratification",
    "psutil": "psutil",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "torch": "torch",
    "torchvision": "torchvision",
    "cloudpickle": "cloudpickle",
}

missing_packages = [
    pip_name
    for module_name, pip_name in PACKAGE_MAP.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_packages
    ])
else:
    print("All required packages are already installed.")

## 1. Imports and reproducibility settings

In [ ]:
import gc
import re
import json
import math
import random
import hashlib
import warnings
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import psutil
from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import BernoulliRBM
from scipy.stats import rankdata

from xgboost import XGBClassifier


In [ ]:
try:
    from iterstrat.ml_stratifiers import (
        MultilabelStratifiedShuffleSplit,
        MultilabelStratifiedKFold,
    )
    ITERATIVE_STRATIFICATION_AVAILABLE = True
except Exception as exc:
    ITERATIVE_STRATIFICATION_AVAILABLE = False
    print("Iterative stratification is unavailable. A stratified fallback will be used.")
    print("Import details:", exc)

In [ ]:
from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import (
    AllChem,
    MACCSkeys,
    Descriptors,
    Draw,
    Crippen,
    Lipinski,
    rdMolDescriptors,
    ChemicalFeatures,
)
from rdkit.Chem.MolStandardize import rdMolStandardize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from torchvision.models import densenet121, DenseNet121_Weights
    TORCHVISION_AVAILABLE = True
except Exception as exc:
    TORCHVISION_AVAILABLE = False
    print("Torchvision is unavailable. The DenseNet hybrid will be skipped.")
    print("Import details:", exc)

In [ ]:
warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

SEED = 42
THRESHOLD = 0.50
CV_FOLDS = 3
TRAIN_RATIO = 0.70
VALID_RATIO = 0.10
TEST_RATIO = 0.20
N_BITS = 2048
RADIUS = 2
TARGET_MEAN_AUC = 0.90
PIPELINE_VERSION = "max_ultra_v1_auc_accuracy_restart_recovery"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
CPU_COUNT = max(1, os.cpu_count() or 1)
N_JOBS = max(1, min(CPU_COUNT - 1, 12))

if RAM_GB < 10:
    RESOURCE_MODE = "low"
elif RAM_GB < 20:
    RESOURCE_MODE = "standard"
else:
    RESOURCE_MODE = "high"

BATCH_TABULAR = 128 if RESOURCE_MODE == "low" else 256
BATCH_IMAGE = 12 if RESOURCE_MODE == "low" else (24 if RESOURCE_MODE == "standard" else 48)
NUM_WORKERS = 0 if platform.system().lower().startswith("win") else min(4, CPU_COUNT // 2)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("=" * 88)
print("Pipeline version:", PIPELINE_VERSION)
print("Random seed:", SEED)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU was not detected. Deep learning models will run on the CPU.")
print(f"RAM: {RAM_GB:.1f} GB | CPU threads: {CPU_COUNT} | Resource mode: {RESOURCE_MODE}")
print("=" * 88)

## 2. Project folders and dataset location

In [ ]:
# Optional manual path example:
# os.environ["TOX21_DATA_PATH"] = r"D:\Drug_Toxicity\datasets\tox21.csv"

candidate_data_paths = []
if os.environ.get("TOX21_DATA_PATH"):
    candidate_data_paths.append(Path(os.environ["TOX21_DATA_PATH"]))

candidate_data_paths += [
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21.csv"),
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21(4).csv"),
    Path("/content/tox21.csv"),
    Path("/content/tox21(4).csv"),
    Path("/mnt/data/tox21(4).csv"),
    Path.cwd() / "datasets" / "tox21.csv",
    Path.cwd() / "datasets" / "tox21(4).csv",
    Path.cwd() / "tox21.csv",
    Path.cwd() / "tox21(4).csv",
]

DATA_PATH = next((path for path in candidate_data_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Tox21 CSV was not found. Set TOX21_DATA_PATH or place the CSV beside the notebook."
    )

In [ ]:
if Path("/content/drive/MyDrive").exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Drug_Toxicity")
elif (Path.cwd() / "datasets").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd() / "Drug_Toxicity_Updated_Shohoj_Pro"

PROJECT_TAG = "Updated_Shohoj_Pro_Max_Ultra"
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / PROJECT_TAG / RUN_STAMP
MODEL_DIR = OUTPUT_DIR / "models"
FEATURE_DIR = OUTPUT_DIR / "features"
FIGURE_DIR = OUTPUT_DIR / "figures"

for folder in [OUTPUT_DIR, MODEL_DIR, FEATURE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
RUN_ID = f"{RAW_SHA256[:10]}_seed{SEED}_{PIPELINE_VERSION}"

print("Dataset:", DATA_PATH)
print("Project root:", PROJECT_ROOT)
print("Run output folder:", OUTPUT_DIR)
print("Run ID:", RUN_ID)

## 3. Load the Tox21 dataset and validate the schema

In [ ]:
EXPECTED_TARGETS = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]

ALIASES = {
    "NR.AhR": "NR-AhR",
    "NR.AR": "NR-AR",
    "NR.AR.LBD": "NR-AR-LBD",
    "NR.Aromatase": "NR-Aromatase",
    "NR.ER": "NR-ER",
    "NR.ER.LBD": "NR-ER-LBD",
    "NR.PPAR.gamma": "NR-PPAR-gamma",
    "SR.ARE": "SR-ARE",
    "SR.ATAD5": "SR-ATAD5",
    "SR.HSE": "SR-HSE",
    "SR.MMP": "SR-MMP",
    "SR.p53": "SR-p53",
}

In [ ]:
df_raw = pd.read_csv(DATA_PATH).rename(columns=ALIASES)

required_columns = set(EXPECTED_TARGETS + ["smiles"])
missing_columns = sorted(required_columns - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

if "mol_id" not in df_raw.columns:
    df_raw["mol_id"] = [f"MOL_{i:06d}" for i in range(len(df_raw))]

for target in EXPECTED_TARGETS:
    df_raw[target] = pd.to_numeric(df_raw[target], errors="coerce")
    invalid_values = set(df_raw[target].dropna().unique()) - {0.0, 1.0}
    if invalid_values:
        raise ValueError(
            f"{target} contains values other than 0, 1 or missing: {invalid_values}"
        )

TARGETS = EXPECTED_TARGETS.copy()

print("Dataset loaded successfully.")
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head(5).style.set_caption("Tox21 Raw Data Preview"))

## 4. Dataset audit: missing labels and class imbalance

In [ ]:
def build_endpoint_summary(frame, targets):
    rows = []
    total_rows = len(frame)

    for target in targets:
        values = frame[target]
        labeled = int(values.notna().sum())
        toxic = int((values == 1).sum())
        non_toxic = int((values == 0).sum())
        missing = total_rows - labeled

        rows.append({
            "Endpoint": target,
            "Labeled": labeled,
            "Toxic (1)": toxic,
            "Non-Toxic (0)": non_toxic,
            "Missing": missing,
            "Missing %": 100.0 * missing / total_rows,
            "Positive Rate %": 100.0 * toxic / labeled if labeled else np.nan,
            "Neg:Pos": non_toxic / max(toxic, 1),
        })

    return pd.DataFrame(rows)

In [ ]:
endpoint_summary_raw = build_endpoint_summary(df_raw, TARGETS)

display(
    endpoint_summary_raw.style
    .format({
        "Missing %": "{:.1f}",
        "Positive Rate %": "{:.1f}",
        "Neg:Pos": "{:.1f}",
    })
    .background_gradient(subset=["Missing %", "Neg:Pos"], cmap="YlOrRd")
    .set_caption("Endpoint-wise Missing Labels and Class Imbalance")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

missing_plot = endpoint_summary_raw.sort_values("Missing %")
axes[0].barh(missing_plot["Endpoint"], missing_plot["Missing %"])
axes[0].set_title("Missing Labels by Endpoint", fontweight="bold")
axes[0].set_xlabel("Missing labels (%)")
for i, value in enumerate(missing_plot["Missing %"]):
    axes[0].text(value + 0.3, i, f"{value:.1f}%", va="center", fontsize=9)

positive_plot = endpoint_summary_raw.sort_values("Positive Rate %")
axes[1].barh(positive_plot["Endpoint"], positive_plot["Positive Rate %"])
axes[1].set_title("Toxic Positive Rate by Endpoint", fontweight="bold")
axes[1].set_xlabel("Positive toxic samples (%)")
for i, value in enumerate(positive_plot["Positive Rate %"]):
    axes[1].text(value + 0.2, i, f"{value:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_dataset_challenges.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 13))

for ax, target in zip(axes.flat, TARGETS):
    counts = (
        df_raw[target]
        .value_counts(dropna=True)
        .reindex([0.0, 1.0], fill_value=0)
    )
    ax.bar(["Non-Toxic", "Toxic"], counts.values)
    ax.set_title(target, fontweight="bold")
    ax.set_ylabel("Count")

    for i, value in enumerate(counts.values):
        ax.text(
            i,
            value + max(counts.values) * 0.015,
            f"{int(value)}",
            ha="center",
            fontsize=8,
        )

plt.suptitle(
    "Per-Task Label Distribution after Missing-Label Masking",
    y=1.02,
    fontweight="bold",
    fontsize=16,
)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_per_task_label_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

## 5. Chemical preprocessing

The preprocessing follows the original notebook logic:

1. Parse each SMILES string.
2. Remove invalid molecules.
3. Keep the largest molecular fragment.
4. Normalize and uncharge the molecule.
5. Sanitize the structure and create canonical SMILES.
6. Remove duplicate canonical molecules.

In [ ]:
largest_fragment = rdMolStandardize.LargestFragmentChooser()
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()


def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, None, "invalid_parse"

        mol = largest_fragment.choose(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)

        canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
        standardized_mol = Chem.MolFromSmiles(canonical_smiles)
        return canonical_smiles, standardized_mol, "ok"

    except Exception as exc:
        return None, None, str(exc)

In [ ]:
df = df_raw.copy()
standardized_records = []

for smiles in tqdm(df["smiles"], desc="Standardizing SMILES"):
    standardized_records.append(standardize_smiles(smiles))

df["canonical_smiles"] = [item[0] for item in standardized_records]
df["mol"] = [item[1] for item in standardized_records]
df["standardize_status"] = [item[2] for item in standardized_records]

invalid_df = df[df["mol"].isna()].copy()
invalid_df.to_csv(OUTPUT_DIR / "invalid_smiles.csv", index=False)

print("Rows before cleaning:", len(df))
print("Invalid SMILES:", int(df["mol"].isna().sum()))

df = df[df["mol"].notna()].copy()
duplicate_count = int(df.duplicated("canonical_smiles").sum())
print("Duplicate canonical SMILES:", duplicate_count)

df = df.drop_duplicates("canonical_smiles").reset_index(drop=True)

save_columns = ["mol_id", "smiles", "canonical_smiles", *TARGETS]
df[save_columns].to_csv(OUTPUT_DIR / "curated_tox21.csv", index=False)

print("Rows after cleaning:", len(df))

In [ ]:
show_count = min(12, len(df))
show_indices = np.linspace(0, len(df) - 1, show_count, dtype=int)
show_molecules = [df.iloc[i]["mol"] for i in show_indices]
show_legends = [str(df.iloc[i]["mol_id"]) for i in show_indices]

molecule_image = Draw.MolsToGridImage(
    show_molecules,
    molsPerRow=4,
    subImgSize=(280, 220),
    legends=show_legends,
)

display(molecule_image)

## 6. Molecular feature engineering

Classical models and the DeepTox-style network use:

- Morgan ECFP4 fingerprint: 2,048 bits, radius 2
- MACCS structural keys: 167 bits
- All available RDKit 2D descriptors

In [ ]:
def morgan_fp(mol, n_bits=N_BITS, radius=RADIUS):
    values = np.zeros((n_bits,), dtype=np.int8)
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits,
    )
    DataStructs.ConvertToNumpyArray(fingerprint, values)
    return values


def maccs_fp(mol):
    values = np.zeros((167,), dtype=np.int8)
    fingerprint = MACCSkeys.GenMACCSKeys(mol)
    DataStructs.ConvertToNumpyArray(fingerprint, values)
    return values

In [ ]:
descriptor_names = [name for name, _ in Descriptors._descList]
descriptor_functions = [function for _, function in Descriptors._descList]


def rdkit_descriptors(mol):
    values = []

    for function in descriptor_functions:
        try:
            values.append(float(function(mol)))
        except Exception:
            values.append(np.nan)

    return values

In [ ]:
print("Generating Morgan ECFP4 fingerprints...")
X_morgan = np.vstack([
    morgan_fp(mol)
    for mol in tqdm(df["mol"], desc="Morgan ECFP4")
]).astype(np.float32)

np.save(FEATURE_DIR / "morgan_2048.npy", X_morgan)
print("Morgan shape:", X_morgan.shape)

In [ ]:
print("Generating MACCS fingerprints...")
X_maccs = np.vstack([
    maccs_fp(mol)
    for mol in tqdm(df["mol"], desc="MACCS")
]).astype(np.float32)

np.save(FEATURE_DIR / "maccs_167.npy", X_maccs)
print("MACCS shape:", X_maccs.shape)

In [ ]:
print("Calculating RDKit descriptors...")
X_desc_array = np.asarray([
    rdkit_descriptors(mol)
    for mol in tqdm(df["mol"], desc="RDKit descriptors")
], dtype=np.float32)

X_desc_raw = pd.DataFrame(
    X_desc_array,
    columns=descriptor_names,
).replace([np.inf, -np.inf], np.nan)

np.save(FEATURE_DIR / "rdkit_descriptors.npy", X_desc_array)
print("Descriptor shape:", X_desc_raw.shape)

In [ ]:
X_fp = np.hstack([X_morgan, X_maccs]).astype(np.float32)
Y = df[TARGETS].to_numpy(dtype=float)

print("Combined binary fingerprint shape:", X_fp.shape)
print("Target matrix shape:", Y.shape)

## 7. Train, validation and test split

The split target is 70% training, 10% validation and 20% testing. Multilabel iterative stratification preserves both positive-label patterns and label-availability patterns across the 12 endpoints.

In [ ]:
all_indices = np.arange(len(df))

Y_positive = np.nan_to_num(Y, nan=0.0).astype(int)
Y_available = np.isfinite(Y).astype(int)
Y_stratification = np.hstack([Y_positive, Y_available]).astype(int)

In [ ]:
if ITERATIVE_STRATIFICATION_AVAILABLE:
    first_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=TEST_RATIO,
        random_state=SEED,
    )

    train_validation_relative, test_relative = next(
        first_split.split(all_indices, Y_stratification)
    )

    train_validation_indices = all_indices[train_validation_relative]
    test_idx = all_indices[test_relative]

    validation_fraction = VALID_RATIO / (TRAIN_RATIO + VALID_RATIO)

    second_split = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED + 1,
    )

    train_relative, validation_relative = next(
        second_split.split(
            train_validation_indices,
            Y_stratification[train_validation_indices],
        )
    )

    train_idx = train_validation_indices[train_relative]
    val_idx = train_validation_indices[validation_relative]
    print("Multilabel iterative stratification was used.")

else:
    row_stratification = (Y_positive.sum(axis=1) > 0).astype(int)

    train_validation_indices, test_idx = train_test_split(
        all_indices,
        test_size=TEST_RATIO,
        random_state=SEED,
        stratify=row_stratification,
    )

    validation_fraction = VALID_RATIO / (TRAIN_RATIO + VALID_RATIO)

    train_idx, val_idx = train_test_split(
        train_validation_indices,
        test_size=validation_fraction,
        random_state=SEED + 1,
        stratify=row_stratification[train_validation_indices],
    )

    print("The stratified fallback split was used.")

Y_train = Y[train_idx]
Y_val = Y[val_idx]
Y_test = Y[test_idx]

print(f"Train:      {len(train_idx):5d} ({100 * len(train_idx) / len(df):.2f}%)")
print(f"Validation: {len(val_idx):5d} ({100 * len(val_idx) / len(df):.2f}%)")
print(f"Test:       {len(test_idx):5d} ({100 * len(test_idx) / len(df):.2f}%)")

In [ ]:
def split_endpoint_counts(indices, split_name):
    rows = []
    split_frame = df.iloc[indices]

    for target in TARGETS:
        rows.append({
            "Split": split_name,
            "Endpoint": target,
            "Non-Toxic": int((split_frame[target] == 0).sum()),
            "Toxic": int((split_frame[target] == 1).sum()),
            "Missing": int(split_frame[target].isna().sum()),
        })

    return pd.DataFrame(rows)


split_distribution = pd.concat([
    split_endpoint_counts(train_idx, "Train"),
    split_endpoint_counts(val_idx, "Validation"),
    split_endpoint_counts(test_idx, "Test"),
], ignore_index=True)

display(
    split_distribution.head(12)
    .style
    .set_caption("Split Distribution Preview")
)

## 8. Train-only descriptor imputation and scaling

The imputer and scaler are fitted only on the training split. Validation and test data are transformed with the training-fitted objects. Target labels are never imputed.

In [ ]:
def build_chemical_features(fit_indices, transform_indices):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    fit_descriptors = imputer.fit_transform(
        X_desc_raw.iloc[fit_indices]
    )
    fit_descriptors = scaler.fit_transform(fit_descriptors)

    transformed_descriptors = imputer.transform(
        X_desc_raw.iloc[transform_indices]
    )
    transformed_descriptors = scaler.transform(
        transformed_descriptors
    )

    X_fit = np.hstack([
        X_fp[fit_indices],
        fit_descriptors,
    ]).astype(np.float32)

    X_transform = np.hstack([
        X_fp[transform_indices],
        transformed_descriptors,
    ]).astype(np.float32)

    return X_fit, X_transform, imputer, scaler

In [ ]:
X_train_chem, X_val_chem, descriptor_imputer, descriptor_scaler = (
    build_chemical_features(train_idx, val_idx)
)

test_descriptors = descriptor_imputer.transform(
    X_desc_raw.iloc[test_idx]
)
test_descriptors = descriptor_scaler.transform(test_descriptors)

X_test_chem = np.hstack([
    X_fp[test_idx],
    test_descriptors,
]).astype(np.float32)

joblib.dump(
    {
        "imputer": descriptor_imputer,
        "scaler": descriptor_scaler,
    },
    MODEL_DIR / "chemical_preprocessing.joblib",
)

print("Train feature shape:", X_train_chem.shape)
print("Validation feature shape:", X_val_chem.shape)
print("Test feature shape:", X_test_chem.shape)

## 9. Evaluation metrics

Only two evaluation metrics are used:

1. **ROC-AUC** — threshold-independent ranking performance.
2. **Accuracy** — classification correctness using the same fixed threshold of **0.50** for every endpoint and every model.

PR-AUC, Balanced Accuracy, Sensitivity, Specificity, F1-score and validation-selected threshold calculations are intentionally removed. Using the fixed 0.50 threshold also avoids the very low balanced-accuracy-selected thresholds that reduced ordinary Accuracy in the previous output. ROC-AUC is unaffected by this threshold change.


In [ ]:
def slugify(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def valid_mask(y_true, y_score=None):
    y_true = np.asarray(y_true, dtype=float)
    mask = np.isfinite(y_true)

    if y_score is not None:
        mask &= np.isfinite(np.asarray(y_score, dtype=float))

    return mask


def safe_auc(y_true, y_score):
    mask = valid_mask(y_true, y_score)
    y = np.asarray(y_true, dtype=float)[mask]
    score = np.asarray(y_score, dtype=float)[mask]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan

    return float(roc_auc_score(y, score))


def mean_metric(values):
    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).to_numpy(dtype=float)

    return (
        float(np.nanmean(values))
        if np.isfinite(values).any()
        else np.nan
    )


def prior_probability(y):
    y = np.asarray(y, dtype=float)
    return float(np.nanmean(y)) if np.isfinite(y).any() else 0.0


In [ ]:
def classification_metrics(
    y_true,
    y_score,
    threshold=THRESHOLD,
):
    mask = valid_mask(y_true, y_score)
    y = np.asarray(y_true, dtype=float)[mask].astype(int)
    score = np.asarray(y_score, dtype=float)[mask]

    if len(y) == 0:
        return {"accuracy": np.nan}

    prediction = (score >= float(threshold)).astype(int)

    return {
        "accuracy": float(accuracy_score(y, prediction)),
    }


In [ ]:
def fixed_threshold_accuracy(y_true, y_score):
    return classification_metrics(
        y_true,
        y_score,
        threshold=THRESHOLD,
    )["accuracy"]


In [ ]:
def predict_positive_probability(model, X):
    if hasattr(model, "predict_proba"):
        probabilities = np.asarray(model.predict_proba(X))

        if probabilities.ndim == 1:
            return probabilities.astype(float)
        if probabilities.shape[1] == 1:
            return np.zeros(len(X), dtype=float)

        return probabilities[:, 1].astype(float)

    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(X), dtype=float)
        score = np.clip(score, -50, 50)
        return 1.0 / (1.0 + np.exp(-score))

    return np.asarray(model.predict(X), dtype=float)

In [ ]:
def endpoint_metric_table(
    P_train_diagnostic,
    P_val,
    P_test,
):
    rows = []

    for j, target in enumerate(TARGETS):
        rows.append({
            "Endpoint": target,
            "Diagnostic Train AUC": safe_auc(
                Y_train[:, j],
                P_train_diagnostic[:, j],
            ),
            "Validation AUC": safe_auc(
                Y_val[:, j],
                P_val[:, j],
            ),
            "Test AUC": safe_auc(
                Y_test[:, j],
                P_test[:, j],
            ),
            "Test Accuracy": fixed_threshold_accuracy(
                Y_test[:, j],
                P_test[:, j],
            ),
        })

    return pd.DataFrame(rows)


In [ ]:
def history_overfit_flag(history):
    if not history:
        return False

    train_loss = np.asarray(history.get("train_loss", []), dtype=float)
    validation_loss = np.asarray(history.get("val_loss", []), dtype=float)

    if len(train_loss) < 8 or len(validation_loss) < 8:
        return False

    best_validation_epoch = int(np.nanargmin(validation_loss))
    tail_size = min(5, len(validation_loss))

    validation_rebound = (
        np.nanmean(validation_loss[-tail_size:])
        > np.nanmin(validation_loss) * 1.08
    )

    comparison_start = max(0, best_validation_epoch - tail_size + 1)
    train_continues_down = (
        np.nanmean(train_loss[-tail_size:])
        < np.nanmean(
            train_loss[comparison_start:best_validation_epoch + 1]
        ) * 0.97
    )

    return bool(validation_rebound and train_continues_down)


def fitting_status(train_auc, validation_auc, history=None):
    if not np.isfinite(train_auc) or not np.isfinite(validation_auc):
        return "Insufficient data"

    gap = train_auc - validation_auc

    if train_auc < 0.70 and validation_auc < 0.70:
        return "Underfitting"
    if validation_auc - train_auc > 0.07:
        return "Split-unstable"
    if gap > 0.10 or (gap > 0.075 and history_overfit_flag(history)):
        return "Overfitting"
    if gap > 0.06:
        return "Mild overfitting"

    return "Well-fitted"

In [ ]:
MODEL_RESULTS = {}
MODEL_ENDPOINT_RESULTS = {}
MODEL_PREDICTIONS = {}
MODEL_HISTORIES = {}
CV_TABLES = {}


def register_model_result(
    model_name,
    P_train_diagnostic,
    P_val,
    P_test,
    history=None,
    notes="",
    diagnostic_method="Diagnostic train prediction",
):
    P_train_diagnostic = np.clip(
        np.nan_to_num(P_train_diagnostic, nan=0.0),
        0.0,
        1.0,
    )
    P_val = np.clip(
        np.nan_to_num(P_val, nan=0.0),
        0.0,
        1.0,
    )
    P_test = np.clip(
        np.nan_to_num(P_test, nan=0.0),
        0.0,
        1.0,
    )

    per_endpoint = endpoint_metric_table(
        P_train_diagnostic,
        P_val,
        P_test,
    )

    train_auc = mean_metric(
        per_endpoint["Diagnostic Train AUC"]
    )
    validation_auc = mean_metric(
        per_endpoint["Validation AUC"]
    )
    test_auc = mean_metric(
        per_endpoint["Test AUC"]
    )
    test_accuracy = mean_metric(
        per_endpoint["Test Accuracy"]
    )

    status = fitting_status(
        train_auc,
        validation_auc,
        history=history,
    )

    summary = {
        "Model": model_name,
        "Diagnostic Train Mean AUC": train_auc,
        "Validation Mean AUC": validation_auc,
        "Test Mean AUC": test_auc,
        "Test Mean Accuracy": test_accuracy,
        "Train-Validation AUC Gap": train_auc - validation_auc,
        "Fitting Status": status,
        "Diagnostic Method": diagnostic_method,
        "Target AUC Reached": bool(
            np.isfinite(test_auc)
            and test_auc >= TARGET_MEAN_AUC
        ),
        "Notes": notes,
    }

    model_output_dir = OUTPUT_DIR / slugify(model_name)
    model_output_dir.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        model_output_dir / "predictions.npz",
        diagnostic_train=P_train_diagnostic,
        validation=P_val,
        test=P_test,
    )

    per_endpoint.to_csv(
        model_output_dir / "per_endpoint_metrics.csv",
        index=False,
    )

    (model_output_dir / "overall_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    if history is not None:
        (model_output_dir / "history.json").write_text(
            json.dumps(history, indent=2),
            encoding="utf-8",
        )
        MODEL_HISTORIES[model_name] = history

    MODEL_RESULTS[model_name] = summary
    MODEL_ENDPOINT_RESULTS[model_name] = per_endpoint
    MODEL_PREDICTIONS[model_name] = {
        "diagnostic_train": P_train_diagnostic,
        "validation": P_val,
        "test": P_test,
    }

    print("\n" + "=" * 96)
    print(model_name)
    print(f"Diagnostic Train Mean AUC : {train_auc:.4f}")
    print(f"Validation Mean AUC       : {validation_auc:.4f}")
    print(f"Test Mean AUC             : {test_auc:.4f}")
    print(f"Test Mean Accuracy        : {test_accuracy:.4f}")
    print(f"Fitting Status            : {status}")
    print(f"Diagnostic Method         : {diagnostic_method}")
    print("=" * 96)

    display(
        per_endpoint[[
            "Endpoint",
            "Diagnostic Train AUC",
            "Validation AUC",
            "Test AUC",
            "Test Accuracy",
        ]]
        .style
        .format({
            "Diagnostic Train AUC": "{:.4f}",
            "Validation AUC": "{:.4f}",
            "Test AUC": "{:.4f}",
            "Test Accuracy": "{:.4f}",
        })
        .background_gradient(
            subset=["Test AUC"],
            cmap="RdYlGn",
            vmin=0.5,
            vmax=1.0,
        )
        .set_caption(
            f"{model_name} — Endpoint-wise Test Performance"
        )
    )

    return summary, per_endpoint


## 10. Classical machine-learning infrastructure

All classical models use 3-fold out-of-fold predictions for the diagnostic training AUC. Final endpoint models are then fitted on the full 70% training split. Validation data are used only for early stopping or model selection, and test labels are not used during training.

In [ ]:
def build_chemical_features_for_fold(fit_indices, transform_indices):
    X_fit, X_transform, _, _ = build_chemical_features(
        fit_indices,
        transform_indices,
    )
    return X_fit, X_transform

In [ ]:
def fit_endpoint_estimator(model, X_train, y_train, X_validation=None, y_validation=None):
    if (
        isinstance(model, XGBClassifier)
        and X_validation is not None
        and y_validation is not None
    ):
        validation_mask = np.isfinite(y_validation)

        if (
            validation_mask.sum() >= 10
            and len(np.unique(y_validation[validation_mask])) == 2
        ):
            model.fit(
                X_train,
                y_train,
                eval_set=[(
                    X_validation[validation_mask],
                    y_validation[validation_mask].astype(int),
                )],
                verbose=False,
            )
            return model

    model.fit(X_train, y_train)
    return model

In [ ]:
def multilabel_cv_splits():
    if ITERATIVE_STRATIFICATION_AVAILABLE:
        fold_targets = np.hstack([
            np.nan_to_num(Y_train, nan=0.0).astype(int),
            np.isfinite(Y_train).astype(int),
        ])

        splitter = MultilabelStratifiedKFold(
            n_splits=CV_FOLDS,
            shuffle=True,
            random_state=SEED,
        )

        return list(splitter.split(
            np.zeros((len(train_idx), 1), dtype=np.float32),
            fold_targets,
        ))

    row_targets = (
        np.nan_to_num(Y_train, nan=0.0).sum(axis=1) > 0
    ).astype(int)

    splitter = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    return list(splitter.split(np.arange(len(train_idx)), row_targets))


CV_SPLITS = multilabel_cv_splits()
print("Prepared", len(CV_SPLITS), "cross-validation folds.")

In [ ]:
def run_endpoint_cv(
    model_name,
    model_factory,
    fold_feature_builder,
):
    fold_rows = []
    P_oof = np.full(
        (len(train_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )

    for fold_number, (train_relative, validation_relative) in enumerate(
        CV_SPLITS,
        start=1,
    ):
        train_absolute = train_idx[train_relative]
        validation_absolute = train_idx[validation_relative]

        X_fold_train, X_fold_validation = fold_feature_builder(
            train_absolute,
            validation_absolute,
        )

        Y_fold_train = Y[train_absolute]
        Y_fold_validation = Y[validation_absolute]

        P_fold_validation = np.full(
            (len(validation_relative), len(TARGETS)),
            np.nan,
            dtype=float,
        )

        for j, target in enumerate(TARGETS):
            y_train_endpoint = Y_fold_train[:, j]
            train_mask = np.isfinite(y_train_endpoint)
            fallback = prior_probability(y_train_endpoint)

            if (
                train_mask.sum() < 10
                or len(np.unique(y_train_endpoint[train_mask])) < 2
            ):
                P_fold_validation[:, j] = fallback
                continue

            model = model_factory(y_train_endpoint[train_mask])
            model = fit_endpoint_estimator(
                model,
                X_fold_train[train_mask],
                y_train_endpoint[train_mask].astype(int),
                X_fold_validation,
                Y_fold_validation[:, j],
            )

            P_fold_validation[:, j] = np.clip(
                predict_positive_probability(model, X_fold_validation),
                0.0,
                1.0,
            )

        P_oof[validation_relative] = P_fold_validation

        fold_auc = mean_metric([
            safe_auc(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
            )
            for j in range(len(TARGETS))
        ])

        fold_accuracy = mean_metric([
            fixed_threshold_accuracy(
                Y_fold_validation[:, j],
                P_fold_validation[:, j],
            )
            for j in range(len(TARGETS))
        ])

        fold_rows.append({
            "Model": model_name,
            "Fold": fold_number,
            "CV Mean AUC": fold_auc,
            "CV Mean Accuracy": fold_accuracy,
        })

        print(
            f"{model_name} | Fold {fold_number}: "
            f"AUC={fold_auc:.4f}, Accuracy={fold_accuracy:.4f}"
        )

    cv_table = pd.DataFrame(fold_rows)
    CV_TABLES[model_name] = cv_table
    cv_table.to_csv(
        OUTPUT_DIR / f"{slugify(model_name)}_three_fold_cv.csv",
        index=False,
    )

    return cv_table, P_oof

In [ ]:
def run_classical_endpoint_model(
    model_name,
    model_factory,
    X_train,
    X_validation,
    X_test,
    fold_feature_builder,
):
    _, P_oof = run_endpoint_cv(
        model_name,
        model_factory,
        fold_feature_builder,
    )

    P_validation = np.full(
        (len(val_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )

    P_test = np.full(
        (len(test_idx), len(TARGETS)),
        np.nan,
        dtype=float,
    )

    model_save_dir = MODEL_DIR / slugify(model_name)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    for j, target in enumerate(TARGETS):
        y_train_endpoint = Y_train[:, j]
        train_mask = np.isfinite(y_train_endpoint)
        fallback = prior_probability(y_train_endpoint)

        if (
            train_mask.sum() < 10
            or len(np.unique(y_train_endpoint[train_mask])) < 2
        ):
            P_validation[:, j] = fallback
            P_test[:, j] = fallback
            continue

        # Endpoint training is intentionally silent for a clean output.
        model = model_factory(y_train_endpoint[train_mask])
        model = fit_endpoint_estimator(
            model,
            X_train[train_mask],
            y_train_endpoint[train_mask].astype(int),
            X_validation,
            Y_val[:, j],
        )

        P_validation[:, j] = np.clip(
            predict_positive_probability(model, X_validation),
            0.0,
            1.0,
        )

        P_test[:, j] = np.clip(
            predict_positive_probability(model, X_test),
            0.0,
            1.0,
        )

        joblib.dump(
            model,
            model_save_dir / f"{j:02d}_{slugify(target)}.joblib",
        )

    return register_model_result(
        model_name,
        P_oof,
        P_validation,
        P_test,
        diagnostic_method="3-fold out-of-fold train prediction",
        notes=(
            "Endpoint-wise regularized model. Final estimators were fitted "
            "on the 70% training split only. Test labels were not used."
        ),
    )

# Classical Machine-Learning Models

The first four models use molecular fingerprints and RDKit descriptors. Random Forest and Extra Trees are bagging-based methods, while XGBoost is a boosting method.

## Model 1 — Regularized Random Forest

Key controls: limited tree depth, minimum leaf size, feature subsampling, row subsampling and balanced bootstrap learning.

In [ ]:
def random_forest_factory(y):
    return RandomForestClassifier(
        n_estimators=900,
        max_depth=18,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features=0.25,
        max_samples=0.85,
        class_weight="balanced_subsample",
        bootstrap=True,
        random_state=SEED,
        n_jobs=N_JOBS,
    )

In [ ]:
rf_summary, rf_per_endpoint = run_classical_endpoint_model(
    "Regularized Random Forest",
    random_forest_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Model 2 — Regularized Extra Trees

Extra Trees adds stronger split randomness than Random Forest. This increases tree diversity and can reduce variance.

In [ ]:
def extra_trees_factory(y):
    return ExtraTreesClassifier(
        n_estimators=1000,
        max_depth=20,
        min_samples_split=8,
        min_samples_leaf=4,
        max_features=0.35,
        bootstrap=True,
        max_samples=0.90,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=N_JOBS,
    )

In [ ]:
et_summary, et_per_endpoint = run_classical_endpoint_model(
    "Regularized Extra Trees",
    extra_trees_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Model 3 — Regularized XGBoost

XGBoost builds trees sequentially. Each new tree focuses on errors left by previous trees. Strong row/feature subsampling, shallow trees, L1/L2 penalties and validation early stopping are used to control overfitting.

In [ ]:
def xgboost_factory(y):
    positive_count = max(int(np.sum(y == 1)), 1)
    negative_count = max(int(np.sum(y == 0)), 1)

    return XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        min_child_weight=6,
        learning_rate=0.015,
        subsample=0.75,
        colsample_bytree=0.70,
        gamma=0.15,
        reg_alpha=0.10,
        reg_lambda=7.0,
        max_delta_step=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=min(negative_count / positive_count, 20.0),
        early_stopping_rounds=80,
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method="hist",
    )

In [ ]:
xgb_summary, xgb_per_endpoint = run_classical_endpoint_model(
    "Regularized XGBoost",
    xgboost_factory,
    X_train_chem,
    X_val_chem,
    X_test_chem,
    build_chemical_features_for_fold,
)

## Model 4 — Regularized SVM with ECFP4 Tanimoto Kernel

The SVM uses exact binary Tanimoto similarity between Morgan ECFP4 fingerprints. The regularization parameter `C` is selected by train-only cross-validation with the one-standard-error rule, which prefers a lower-complexity value when performance is statistically similar.

In [ ]:
def compute_tanimoto_kernel(A, B, block_size=256):
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    kernel = np.empty((len(A), len(B)), dtype=np.float32)
    B_sum = B.sum(axis=1)[None, :]

    for start in tqdm(
        range(0, len(A), block_size),
        desc="Computing Tanimoto kernel",
    ):
        end = min(start + block_size, len(A))
        block = A[start:end]
        intersection = block @ B.T
        denominator = block.sum(axis=1)[:, None] + B_sum - intersection

        kernel[start:end] = np.divide(
            intersection,
            denominator,
            out=np.zeros_like(intersection),
            where=denominator > 0,
        )

    return kernel

In [ ]:
X_train_ecfp = X_morgan[train_idx].astype(np.float32)
X_val_ecfp = X_morgan[val_idx].astype(np.float32)
X_test_ecfp = X_morgan[test_idx].astype(np.float32)

K_train = compute_tanimoto_kernel(X_train_ecfp, X_train_ecfp)
K_validation = compute_tanimoto_kernel(X_val_ecfp, X_train_ecfp)
K_test = compute_tanimoto_kernel(X_test_ecfp, X_train_ecfp)

print("Train kernel shape:", K_train.shape)
print("Validation kernel shape:", K_validation.shape)
print("Test kernel shape:", K_test.shape)

In [ ]:
def select_tanimoto_c(K, y, c_grid=None):
    if c_grid is None:
        c_grid = (
            [0.10, 0.25, 0.50, 1.0, 2.0]
            if RESOURCE_MODE == "low"
            else [0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0]
        )

    K = np.asarray(K)
    y = np.asarray(y).astype(int)

    splitter = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED,
    )

    rows = []

    for c_value in c_grid:
        fold_scores = []

        for train_relative, validation_relative in splitter.split(
            np.arange(len(y)),
            y,
        ):
            model = SVC(
                kernel="precomputed",
                C=float(c_value),
                class_weight="balanced",
                probability=False,
                cache_size=1500,
            )

            model.fit(
                K[np.ix_(train_relative, train_relative)],
                y[train_relative],
            )

            score = model.decision_function(
                K[np.ix_(validation_relative, train_relative)]
            )

            fold_scores.append(
                safe_auc(y[validation_relative], score)
            )

        rows.append({
            "C": float(c_value),
            "Mean AUC": mean_metric(fold_scores),
            "Std AUC": float(np.nanstd(fold_scores, ddof=1)),
        })

    cv_table = pd.DataFrame(rows).sort_values("C").reset_index(drop=True)

    best_position = int(cv_table["Mean AUC"].to_numpy().argmax())
    best_row = cv_table.iloc[best_position]
    best_mean = float(best_row["Mean AUC"])
    best_standard_error = float(best_row["Std AUC"]) / math.sqrt(3)
    cutoff = best_mean - best_standard_error

    eligible = cv_table[cv_table["Mean AUC"] >= cutoff]
    selected = eligible.sort_values("C").iloc[0]

    return float(selected["C"]), float(selected["Mean AUC"]), cv_table

In [ ]:
def tanimoto_oof_predictions(K_train, y_full, selected_c):
    y_full = np.asarray(y_full, dtype=float)
    labeled_indices = np.where(np.isfinite(y_full))[0]
    labeled_y = y_full[labeled_indices].astype(int)

    fallback = prior_probability(y_full)
    prediction = np.full(len(y_full), fallback, dtype=float)

    if len(labeled_y) < 15 or len(np.unique(labeled_y)) < 2:
        return prediction

    splitter = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED,
    )

    for train_relative, validation_relative in splitter.split(
        np.arange(len(labeled_y)),
        labeled_y,
    ):
        train_positions = labeled_indices[train_relative]
        validation_positions = labeled_indices[validation_relative]

        model = SVC(
            kernel="precomputed",
            C=selected_c,
            class_weight="balanced",
            probability=False,
            cache_size=1500,
        )

        model.fit(
            K_train[np.ix_(train_positions, train_positions)],
            y_full[train_positions].astype(int),
        )

        score = model.decision_function(
            K_train[np.ix_(validation_positions, train_positions)]
        )

        prediction[validation_positions] = 1.0 / (
            1.0 + np.exp(-np.clip(score, -50, 50))
        )

    return prediction

In [ ]:
def run_tanimoto_svm(model_name, K_train, K_validation, K_test):
    P_oof = np.zeros((len(train_idx), len(TARGETS)), dtype=float)
    P_validation = np.zeros((len(val_idx), len(TARGETS)), dtype=float)
    P_test = np.zeros((len(test_idx), len(TARGETS)), dtype=float)

    selected_rows = []
    model_save_dir = MODEL_DIR / slugify(model_name)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    for j, target in enumerate(TARGETS):
        y_train_endpoint = Y_train[:, j]
        labeled_indices = np.where(np.isfinite(y_train_endpoint))[0]
        labeled_y = y_train_endpoint[labeled_indices].astype(int)
        fallback = prior_probability(y_train_endpoint)

        if len(labeled_y) < 15 or len(np.unique(labeled_y)) < 2:
            P_oof[:, j] = fallback
            P_validation[:, j] = fallback
            P_test[:, j] = fallback
            selected_rows.append({
                "Endpoint": target,
                "C": 0.5,
                "CV AUC": np.nan,
            })
            continue

        labeled_kernel = K_train[np.ix_(labeled_indices, labeled_indices)]
        best_c, cv_auc, cv_table = select_tanimoto_c(
            labeled_kernel,
            labeled_y,
        )

        cv_table.to_csv(
            OUTPUT_DIR / f"svm_{slugify(target)}_c_search.csv",
            index=False,
        )

        print(
            f"{target}: one-SE selected C={best_c:g}, CV AUC={cv_auc:.4f}"
        )

        P_oof[:, j] = tanimoto_oof_predictions(
            K_train,
            y_train_endpoint,
            best_c,
        )

        model = SVC(
            kernel="precomputed",
            C=best_c,
            class_weight="balanced",
            probability=False,
            cache_size=1500,
            random_state=SEED,
        )

        model.fit(
            K_train[np.ix_(labeled_indices, labeled_indices)],
            labeled_y,
        )

        validation_score = model.decision_function(
            K_validation[:, labeled_indices]
        )
        test_score = model.decision_function(
            K_test[:, labeled_indices]
        )

        P_validation[:, j] = 1.0 / (
            1.0 + np.exp(-np.clip(validation_score, -50, 50))
        )
        P_test[:, j] = 1.0 / (
            1.0 + np.exp(-np.clip(test_score, -50, 50))
        )

        joblib.dump(
            model,
            model_save_dir / f"{j:02d}_{slugify(target)}.joblib",
        )

        selected_rows.append({
            "Endpoint": target,
            "C": best_c,
            "CV AUC": cv_auc,
        })

    pd.DataFrame(selected_rows).to_csv(
        OUTPUT_DIR / "svm_selected_c.csv",
        index=False,
    )

    result = register_model_result(
        model_name,
        P_oof,
        P_validation,
        P_test,
        diagnostic_method="3-fold OOF Tanimoto-SVM prediction",
        notes=(
            "Exact ECFP4 Tanimoto kernel. C was selected with train-only "
            "cross-validation and the one-standard-error rule."
        ),
    )

    return result, {
        row["Endpoint"]: row["C"]
        for row in selected_rows
    }

In [ ]:
(
    svm_tanimoto_result,
    TANIMOTO_C_BY_ENDPOINT,
) = run_tanimoto_svm(
    "Regularized SVM Tanimoto",
    K_train,
    K_validation,
    K_test,
)

svm_tanimoto_summary, svm_tanimoto_per_endpoint = svm_tanimoto_result

# Deep-Learning Infrastructure

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.as_tensor(
            np.asarray(X),
            dtype=torch.float32,
        )
        self.Y = torch.as_tensor(
            np.asarray(Y),
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.Y[index]

In [ ]:
def compute_pos_weight(Y_matrix, cap=10.0, power=0.75):
    weights = []

    for j in range(Y_matrix.shape[1]):
        y = Y_matrix[:, j]
        positive = max(int(np.nansum(y == 1)), 1)
        negative = max(int(np.nansum(y == 0)), 1)

        ratio = (negative / positive) ** power
        weights.append(min(ratio, cap))

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

In [ ]:
def masked_weighted_bce(
    logits,
    targets,
    pos_weight=None,
    label_smoothing=0.01,
):
    mask = torch.isfinite(targets)
    safe_targets = torch.nan_to_num(targets, nan=0.0)

    if label_smoothing > 0:
        safe_targets = (
            safe_targets * (1.0 - label_smoothing)
            + 0.5 * label_smoothing
        )

    loss = F.binary_cross_entropy_with_logits(
        logits,
        safe_targets,
        reduction="none",
        pos_weight=pos_weight,
    )

    return (
        (loss * mask.float()).sum()
        / mask.float().sum().clamp_min(1.0)
    )

In [ ]:
def masked_focal_bce(
    logits,
    targets,
    pos_weight=None,
    gamma=1.0,
    label_smoothing=0.01,
):
    mask = torch.isfinite(targets)
    safe_targets = torch.nan_to_num(targets, nan=0.0)

    if label_smoothing > 0:
        safe_targets = (
            safe_targets * (1.0 - label_smoothing)
            + 0.5 * label_smoothing
        )

    bce = F.binary_cross_entropy_with_logits(
        logits,
        safe_targets,
        reduction="none",
        pos_weight=pos_weight,
    )

    probability = torch.sigmoid(logits)
    pt = (
        safe_targets * probability
        + (1.0 - safe_targets) * (1.0 - probability)
    )

    focal_factor = (1.0 - pt).clamp_min(1e-6).pow(gamma)
    loss = bce * focal_factor

    return (
        (loss * mask.float()).sum()
        / mask.float().sum().clamp_min(1.0)
    )

In [ ]:
def filter_sparse_binary_features(
    X_train,
    X_validation,
    X_test,
    binary_dimension,
    min_count=8,
    max_fraction=0.995,
):
    X_train = np.asarray(X_train)
    X_validation = np.asarray(X_validation)
    X_test = np.asarray(X_test)

    binary_train = X_train[:, :binary_dimension]
    count = np.sum(binary_train > 0.5, axis=0)

    keep = (
        (count >= min_count)
        & (count <= int(len(X_train) * max_fraction))
    )

    X_train_new = np.hstack([
        X_train[:, :binary_dimension][:, keep],
        X_train[:, binary_dimension:],
    ])

    X_validation_new = np.hstack([
        X_validation[:, :binary_dimension][:, keep],
        X_validation[:, binary_dimension:],
    ])

    X_test_new = np.hstack([
        X_test[:, :binary_dimension][:, keep],
        X_test[:, binary_dimension:],
    ])

    print(
        f"Sparse binary features: {binary_dimension} -> {int(keep.sum())} retained"
    )

    return (
        X_train_new.astype(np.float32),
        X_validation_new.astype(np.float32),
        X_test_new.astype(np.float32),
        keep,
    )

In [ ]:
@torch.no_grad()
def predict_tabular_torch(
    model,
    X,
    batch_size=512,
    probability_function=None,
):
    model.eval()

    loader = DataLoader(
        torch.as_tensor(np.asarray(X), dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False,
    )

    predictions = []

    for X_batch in loader:
        raw_output = model(X_batch.to(DEVICE))

        if probability_function is None:
            probability = torch.sigmoid(raw_output)
        else:
            probability = probability_function(raw_output)

        predictions.append(
            probability.detach().cpu().numpy()
        )

    return np.vstack(predictions)

In [ ]:
def clone_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def average_state_dicts(state_dicts):
    if len(state_dicts) == 1:
        return state_dicts[0]

    averaged = {}

    for key in state_dicts[0]:
        values = [state[key] for state in state_dicts]

        if torch.is_floating_point(values[0]):
            averaged[key] = torch.stack(values, dim=0).mean(dim=0)
        else:
            averaged[key] = values[0]

    return averaged

In [ ]:
def train_tabular_multitask(
    model_name,
    model,
    X_train,
    Y_train_local,
    X_validation,
    Y_validation_local,
    X_test,
    optimizer,
    max_epochs,
    batch_size,
    patience,
    loss_function,
    scheduler=None,
    probability_function=None,
    notes="",
    top_k=3,
    min_delta=2e-4,
    minimum_epochs=10,
):
    train_loader = DataLoader(
        TabularDataset(X_train, Y_train_local),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    validation_loader = DataLoader(
        TabularDataset(X_validation, Y_validation_local),
        batch_size=max(batch_size, 256),
        shuffle=False,
        num_workers=0,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": [],
        "learning_rate": [],
    }

    best_validation_auc = -np.inf
    wait = 0
    top_states = []

    model = model.to(DEVICE)

    for epoch in range(max_epochs):
        model.train()
        train_losses = []

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            raw_output = model(X_batch)
            loss = loss_function(raw_output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )

            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        validation_probabilities = []
        validation_targets = []

        with torch.no_grad():
            for X_batch, Y_batch in validation_loader:
                X_batch = X_batch.to(DEVICE)
                Y_batch = Y_batch.to(DEVICE)

                raw_output = model(X_batch)
                validation_losses.append(
                    float(loss_function(raw_output, Y_batch).detach().cpu())
                )

                if probability_function is None:
                    probability = torch.sigmoid(raw_output)
                else:
                    probability = probability_function(raw_output)

                validation_probabilities.append(
                    probability.cpu().numpy()
                )
                validation_targets.append(
                    Y_batch.cpu().numpy()
                )

        train_loss = float(np.mean(train_losses))
        validation_loss = float(np.mean(validation_losses))
        validation_probability = np.vstack(validation_probabilities)
        validation_target = np.vstack(validation_targets)

        validation_auc = mean_metric([
            safe_auc(
                validation_target[:, j],
                validation_probability[:, j],
            )
            for j in range(len(TARGETS))
        ])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation_loss)
        history["val_auc"].append(validation_auc)
        history["learning_rate"].append(
            float(optimizer.param_groups[0]["lr"])
        )

        if scheduler is not None:
            try:
                scheduler.step(validation_auc)
            except TypeError:
                scheduler.step()

        score = validation_auc if np.isfinite(validation_auc) else -np.inf

        top_states.append({
            "epoch": epoch + 1,
            "val_auc": float(score),
            "state_dict": clone_state_dict(model),
        })

        top_states = sorted(
            top_states,
            key=lambda item: item["val_auc"],
            reverse=True,
        )[:top_k]

        if score > best_validation_auc + min_delta:
            best_validation_auc = score
            wait = 0
        else:
            wait += 1

        print(
            f"Epoch {epoch + 1:03d}/{max_epochs} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={validation_loss:.5f} | "
            f"val_auc={validation_auc:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if epoch + 1 >= minimum_epochs and wait >= patience:
            print(
                f"Validation-AUC early stopping after {patience} "
                "epochs without sufficient improvement."
            )
            break

    valid_top_states = [
        item
        for item in top_states
        if np.isfinite(item["val_auc"])
    ]

    if valid_top_states:
        averaged_state = average_state_dicts([
            item["state_dict"]
            for item in valid_top_states
        ])
        model.load_state_dict(averaged_state)

        print(
            "Top checkpoint epochs:",
            [
                (item["epoch"], round(item["val_auc"], 4))
                for item in valid_top_states
            ],
        )

    model = model.to(DEVICE)

    P_train = predict_tabular_torch(
        model,
        X_train,
        probability_function=probability_function,
    )

    P_validation = predict_tabular_torch(
        model,
        X_validation,
        probability_function=probability_function,
    )

    P_test = predict_tabular_torch(
        model,
        X_test,
        probability_function=probability_function,
    )

    history["best_val_auc"] = float(best_validation_auc)
    history["top_checkpoint_records"] = [
        {
            "epoch": item["epoch"],
            "val_auc": item["val_auc"],
        }
        for item in valid_top_states
    ]

    model_path = MODEL_DIR / f"{slugify(model_name)}.pt"
    torch.save(model.state_dict(), model_path)

    result = register_model_result(
        model_name,
        P_train,
        P_validation,
        P_test,
        history=history,
        diagnostic_method=(
            "Best-checkpoint train prediction; model selected by validation "
            "AUC and top-checkpoint weight averaging"
        ),
        notes=notes,
    )

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

## Model 5 — Strongly Regularized DeepTox-style Multi-task DNN

The original notebook showed residual overfitting in the DeepTox-style model. The shared network is therefore reduced from `1024-512-256` to `512-256-128`, weight decay is increased, rare binary features are filtered more strictly, and a mild focal term is used with capped class weights.

In [ ]:
(
    X_train_deeptox_raw,
    X_val_deeptox_raw,
    X_test_deeptox_raw,
    DEEPTOX_KEEP_MASK,
) = filter_sparse_binary_features(
    X_train_chem,
    X_val_chem,
    X_test_chem,
    binary_dimension=N_BITS + 167,
    min_count=8,
    max_fraction=0.995,
)

X_train_deeptox = np.tanh(X_train_deeptox_raw).astype(np.float32)
X_val_deeptox = np.tanh(X_val_deeptox_raw).astype(np.float32)
X_test_deeptox = np.tanh(X_test_deeptox_raw).astype(np.float32)

np.save(
    MODEL_DIR / "deeptox_sparse_binary_keep_mask.npy",
    DEEPTOX_KEEP_MASK,
)

In [ ]:
class DeepToxMTDNN(nn.Module):
    def __init__(self, input_dimension, number_of_tasks):
        super().__init__()

        self.input_dropout = nn.Dropout(0.15)

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.45),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.40),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(128, number_of_tasks),
        )

    def forward(self, X):
        return self.network(self.input_dropout(X))

In [ ]:
deeptox_model = DeepToxMTDNN(
    X_train_deeptox.shape[1],
    len(TARGETS),
).to(DEVICE)

pos_weight_deeptox = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.70,
)

deeptox_optimizer = torch.optim.AdamW(
    deeptox_model.parameters(),
    lr=2.5e-4,
    weight_decay=5e-4,
)

deeptox_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    deeptox_optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
)

In [ ]:
deeptox_summary, deeptox_per_endpoint = train_tabular_multitask(
    "Strongly Regularized DeepTox-style MT-DNN",
    deeptox_model,
    X_train_deeptox,
    Y_train,
    X_val_deeptox,
    Y_val,
    X_test_deeptox,
    optimizer=deeptox_optimizer,
    max_epochs=160,
    batch_size=512 if RESOURCE_MODE != "low" else 256,
    patience=18,
    minimum_epochs=15,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=pos_weight_deeptox,
        gamma=0.75,
        label_smoothing=0.015,
    ),
    scheduler=deeptox_scheduler,
    top_k=3,
    notes=(
        "Reduced 512-256-128 shared network with ECFP4, MACCS and RDKit "
        "descriptors; strict sparse-feature filtering, tanh transformation, "
        "LayerNorm, dropout, stronger AdamW weight decay, masked focal BCE, "
        "validation-AUC early stopping and in-memory checkpoint averaging."
    ),
)

## Model 6 — Multitask CapsNet with RBM Pretraining and Dynamic Routing

The CapsNet branch retains the paper-inspired 179-feature input and two RBMs. The original margin-loss implementation underfit on the captured run, so this revised version optimizes an imbalance-aware focal BCE on the difference between positive and negative capsule lengths. Dynamic routing is retained.

In [ ]:
def esol_estimate(mol):
    logp = Crippen.MolLogP(mol)
    molecular_weight = Descriptors.MolWt(mol)
    rotatable_bonds = Lipinski.NumRotatableBonds(mol)
    heavy_atoms = max(mol.GetNumHeavyAtoms(), 1)
    aromatic_atoms = sum(atom.GetIsAromatic() for atom in mol.GetAtoms())
    aromatic_fraction = aromatic_atoms / heavy_atoms

    return (
        0.16
        - 1.5 * logp
        - 0.01 * (molecular_weight - 40.0)
        + 0.066 * rotatable_bonds
        + 0.066 * aromatic_fraction
    )


def capsnet_13_descriptors(mol):
    logp = Crippen.MolLogP(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    labute_area = rdMolDescriptors.CalcLabuteASA(mol)

    return [
        logp,
        logp,
        esol_estimate(mol),
        Descriptors.MolWt(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumRotatableBonds(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        sum(
            atom.GetAtomicNum() in (7, 8)
            for atom in mol.GetAtoms()
        ),
        tpsa,
        tpsa / max(labute_area, 1e-6),
        labute_area,
    ]

In [ ]:
X_caps_all = np.vstack([
    np.concatenate([
        maccs_fp(mol)[1:],
        np.asarray(capsnet_13_descriptors(mol), dtype=np.float32),
    ])
    for mol in tqdm(df["mol"], desc="CapsNet features")
]).astype(np.float32)

caps_scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
X_train_caps = caps_scaler.fit_transform(
    X_caps_all[train_idx]
).astype(np.float32)
X_val_caps = caps_scaler.transform(
    X_caps_all[val_idx]
).astype(np.float32)
X_test_caps = caps_scaler.transform(
    X_caps_all[test_idx]
).astype(np.float32)

joblib.dump(
    caps_scaler,
    MODEL_DIR / "capsnet_minmax_scaler.joblib",
)

print("CapsNet input shape:", X_train_caps.shape)

In [ ]:
print("Pretraining RBM 1: 179 -> 256")
rbm1 = BernoulliRBM(
    n_components=256,
    learning_rate=0.01,
    batch_size=148,
    n_iter=20 if RESOURCE_MODE == "low" else 30,
    verbose=1,
    random_state=SEED,
)

H1 = rbm1.fit_transform(X_train_caps)

print("Pretraining RBM 2: 256 -> 128")
rbm2 = BernoulliRBM(
    n_components=128,
    learning_rate=0.01,
    batch_size=148,
    n_iter=15 if RESOURCE_MODE == "low" else 25,
    verbose=1,
    random_state=SEED,
)

rbm2.fit(H1)

joblib.dump(rbm1, MODEL_DIR / "capsnet_rbm1.joblib")
joblib.dump(rbm2, MODEL_DIR / "capsnet_rbm2.joblib")

In [ ]:
def squash_capsules(values, dimension=-1, epsilon=1e-8):
    norm_squared = (
        values * values
    ).sum(dim=dimension, keepdim=True)

    scale = (
        norm_squared
        / (1.0 + norm_squared)
        / torch.sqrt(norm_squared + epsilon)
    )

    return scale * values

In [ ]:
class MultitaskCapsNetRBM(nn.Module):
    def __init__(
        self,
        input_dimension=179,
        number_of_tasks=12,
        routing_iterations=2,
        primary_capsules=16,
        primary_dimension=8,
        class_dimension=4,
    ):
        super().__init__()

        self.number_of_tasks = number_of_tasks
        self.routing_iterations = routing_iterations
        self.primary_capsules = primary_capsules
        self.primary_dimension = primary_dimension

        self.dropout = nn.Dropout(0.15)
        self.rbm1 = nn.Linear(input_dimension, 256)
        self.rbm2 = nn.Linear(256, 128)

        self.primary = nn.Linear(
            128,
            primary_capsules * primary_dimension,
        )

        self.route_weights = nn.Parameter(
            0.02 * torch.randn(
                number_of_tasks,
                primary_capsules,
                2,
                class_dimension,
                primary_dimension,
            )
        )

    def forward(self, X):
        X = torch.sigmoid(self.rbm1(self.dropout(X)))
        X = torch.sigmoid(self.rbm2(self.dropout(X)))

        primary = self.primary(X).view(
            -1,
            self.primary_capsules,
            self.primary_dimension,
        )

        primary = squash_capsules(primary, dimension=-1)

        predicted_capsules = torch.einsum(
            "bni,tnodi->btnod",
            primary,
            self.route_weights,
        )

        routing_logits = torch.zeros(
            predicted_capsules.shape[:-1],
            device=X.device,
            dtype=X.dtype,
        )

        for routing_step in range(self.routing_iterations):
            coupling = torch.softmax(routing_logits, dim=-1)

            combined = (
                coupling.unsqueeze(-1) * predicted_capsules
            ).sum(dim=2)

            class_capsules = squash_capsules(
                combined,
                dimension=-1,
            )

            if routing_step < self.routing_iterations - 1:
                agreement = (
                    predicted_capsules
                    * class_capsules.unsqueeze(2)
                ).sum(dim=-1)

                routing_logits = routing_logits + agreement

        capsule_lengths = torch.linalg.vector_norm(
            class_capsules,
            dim=-1,
        )

        return 6.0 * (
            capsule_lengths[..., 1]
            - capsule_lengths[..., 0]
        )

In [ ]:
caps_model = MultitaskCapsNetRBM(
    number_of_tasks=len(TARGETS),
    routing_iterations=2,
).to(DEVICE)

with torch.no_grad():
    caps_model.rbm1.weight.copy_(
        torch.tensor(
            rbm1.components_,
            dtype=torch.float32,
            device=DEVICE,
        )
    )
    caps_model.rbm1.bias.copy_(
        torch.tensor(
            rbm1.intercept_hidden_,
            dtype=torch.float32,
            device=DEVICE,
        )
    )
    caps_model.rbm2.weight.copy_(
        torch.tensor(
            rbm2.components_,
            dtype=torch.float32,
            device=DEVICE,
        )
    )
    caps_model.rbm2.bias.copy_(
        torch.tensor(
            rbm2.intercept_hidden_,
            dtype=torch.float32,
            device=DEVICE,
        )
    )

In [ ]:
caps_pos_weight = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.55,
)

caps_optimizer = torch.optim.AdamW(
    caps_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

caps_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    caps_optimizer,
    mode="max",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
)

CAPS_EPOCHS = 180 if RESOURCE_MODE == "low" else 260

In [ ]:
caps_summary, caps_per_endpoint = train_tabular_multitask(
    "Regularized Multitask CapsNet + RBM",
    caps_model,
    X_train_caps,
    Y_train,
    X_val_caps,
    Y_val,
    X_test_caps,
    optimizer=caps_optimizer,
    max_epochs=CAPS_EPOCHS,
    batch_size=148,
    patience=30,
    minimum_epochs=20,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=caps_pos_weight,
        gamma=1.0,
        label_smoothing=0.01,
    ),
    scheduler=caps_scheduler,
    top_k=3,
    notes=(
        "Paper-inspired 179-feature input with two RBMs and dynamic routing. "
        "The revised supervised objective uses capped imbalance weights and "
        "focal BCE on positive-versus-negative capsule evidence to address "
        "the underfitting observed in the previous margin-loss run."
    ),
)

## Model 7 — Boosted DenseNet121 + Chemical XGBoost Hybrid

This is the revised hybrid model. It now performs **feature-level fusion**:

`ECFP4 + MACCS + standardized RDKit descriptors + DenseNet121 visual PCA features -> regularized XGBoost`

The final classifier is therefore a **boosting model**, not an SVM. DenseNet121 is used only as a pretrained molecular-image feature extractor. PCA and scaling are fitted on training data only.

In [ ]:
class MoleculeImageDataset(Dataset):
    def __init__(self, smiles_list, transform):
        self.smiles_list = list(smiles_list)
        self.transform = transform

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, index):
        mol = Chem.MolFromSmiles(self.smiles_list[index])
        image = Draw.MolToImage(
            mol,
            size=(224, 224),
        ).convert("RGB")

        return self.transform(image)

In [ ]:
DENSENET_READY = False
X_dense_all = None

if not TORCHVISION_AVAILABLE:
    print("DenseNet121 hybrid skipped because torchvision is unavailable.")
else:
    try:
        dense_weights = DenseNet121_Weights.DEFAULT
        dense_transform = dense_weights.transforms()
        dense_backbone = densenet121(weights=dense_weights)
        dense_backbone.classifier = nn.Identity()
        dense_backbone = dense_backbone.to(DEVICE).eval()
        DENSENET_READY = True
        print("ImageNet-pretrained DenseNet121 loaded successfully.")

    except Exception as exc:
        print("DenseNet121 pretrained weights could not be loaded.")
        print("The hybrid model will be skipped rather than using random visual features.")
        print("Details:", exc)

In [ ]:
if DENSENET_READY:
    image_dataset = MoleculeImageDataset(
        df["canonical_smiles"].tolist(),
        dense_transform,
    )

    image_loader = DataLoader(
        image_dataset,
        batch_size=BATCH_IMAGE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    dense_feature_batches = []

    with torch.inference_mode():
        for image_batch in tqdm(
            image_loader,
            desc="DenseNet121 feature extraction",
        ):
            features = dense_backbone(
                image_batch.to(DEVICE)
            )

            dense_feature_batches.append(
                features.cpu().numpy().astype(np.float32)
            )

    X_dense_all = np.vstack(dense_feature_batches)
    np.save(
        FEATURE_DIR / "densenet121_features_1024.npy",
        X_dense_all,
    )

    print("DenseNet feature shape:", X_dense_all.shape)

In [ ]:
def fit_visual_transformer(fit_indices, transform_indices):
    number_of_components = min(
        64,
        len(fit_indices) - 1,
        X_dense_all.shape[1],
    )

    pca = PCA(
        n_components=number_of_components,
        random_state=SEED,
        svd_solver="randomized",
        whiten=True,
    )

    Z_fit = pca.fit_transform(
        X_dense_all[fit_indices]
    )

    visual_scaler = StandardScaler()
    Z_fit = visual_scaler.fit_transform(Z_fit)

    Z_transform = visual_scaler.transform(
        pca.transform(X_dense_all[transform_indices])
    )

    return (
        Z_fit.astype(np.float32),
        Z_transform.astype(np.float32),
        pca,
        visual_scaler,
    )

In [ ]:
def build_hybrid_features_for_fold(fit_indices, transform_indices):
    chemical_fit, chemical_transform = (
        build_chemical_features_for_fold(
            fit_indices,
            transform_indices,
        )
    )

    visual_fit, visual_transform, _, _ = fit_visual_transformer(
        fit_indices,
        transform_indices,
    )

    X_fit = np.hstack([
        chemical_fit,
        visual_fit,
    ]).astype(np.float32)

    X_transform = np.hstack([
        chemical_transform,
        visual_transform,
    ]).astype(np.float32)

    return X_fit, X_transform

In [ ]:
if DENSENET_READY:
    Z_train, Z_val, dense_pca, dense_scaler = fit_visual_transformer(
        train_idx,
        val_idx,
    )

    Z_test = dense_scaler.transform(
        dense_pca.transform(X_dense_all[test_idx])
    ).astype(np.float32)

    X_train_hybrid = np.hstack([
        X_train_chem,
        Z_train,
    ]).astype(np.float32)

    X_val_hybrid = np.hstack([
        X_val_chem,
        Z_val,
    ]).astype(np.float32)

    X_test_hybrid = np.hstack([
        X_test_chem,
        Z_test,
    ]).astype(np.float32)

    joblib.dump(
        {
            "pca": dense_pca,
            "scaler": dense_scaler,
        },
        MODEL_DIR / "densenet_visual_transformer.joblib",
    )

    print("Hybrid feature shape:", X_train_hybrid.shape)

In [ ]:
def hybrid_xgboost_factory(y):
    positive_count = max(int(np.sum(y == 1)), 1)
    negative_count = max(int(np.sum(y == 0)), 1)

    return XGBClassifier(
        n_estimators=3000,
        max_depth=2,
        min_child_weight=8,
        learning_rate=0.015,
        subsample=0.75,
        colsample_bytree=0.65,
        gamma=0.20,
        reg_alpha=0.15,
        reg_lambda=10.0,
        max_delta_step=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=min(negative_count / positive_count, 15.0),
        early_stopping_rounds=100,
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method="hist",
    )

In [ ]:
if DENSENET_READY:
    hybrid_summary, hybrid_per_endpoint = run_classical_endpoint_model(
        "Boosted DenseNet121 + Chemical XGBoost Hybrid",
        hybrid_xgboost_factory,
        X_train_hybrid,
        X_val_hybrid,
        X_test_hybrid,
        build_hybrid_features_for_fold,
    )
else:
    print("Boosted DenseNet121 hybrid was not executed.")

## Model 8 — Regularized Multi-channel 2D CNN

Each molecule is represented as six 48 x 48 physicochemical grids: hydrogen bonding, hydrophobicity, metallicity, excluded volume, positive ionization and negative ionization. The implementation remains an RDKit-based open-source approximation of the paper workflow.

In [ ]:
GRID_SIZE = 48
GRID_HALF = 12.0
GRID_AXIS = np.linspace(
    -GRID_HALF,
    GRID_HALF,
    GRID_SIZE,
    dtype=np.float32,
)

GRID_X, GRID_Y = np.meshgrid(
    GRID_AXIS,
    GRID_AXIS,
    indexing="xy",
)

PERIODIC_TABLE = Chem.GetPeriodicTable()
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(
    str(Path(RDConfig.RDDataDir) / "BaseFeatures.fdef")
)

METALS = {
    3, 4, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 27,
    28, 29, 30, 31, 37, 38, 47, 48, 55, 56, 78, 79, 80,
}

HYDROPHOBIC_ATOMS = {6, 9, 16, 17, 35, 53}

In [ ]:
def molecule_six_channel_grid(mol):
    mol = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol)

    conformer = mol.GetConformer()
    coordinates = np.asarray([
        [
            conformer.GetAtomPosition(i).x,
            conformer.GetAtomPosition(i).y,
        ]
        for i in range(mol.GetNumAtoms())
    ], dtype=np.float32)

    coordinates -= coordinates.mean(axis=0, keepdims=True)

    span = max(
        np.ptp(coordinates[:, 0]),
        np.ptp(coordinates[:, 1]),
        1.0,
    )

    coordinates *= min(1.0, 20.0 / span)

    donor_ids = set()
    acceptor_ids = set()

    for feature in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if feature.GetFamily() == "Donor":
            donor_ids.update(feature.GetAtomIds())
        if feature.GetFamily() == "Acceptor":
            acceptor_ids.update(feature.GetAtomIds())

    channels = np.zeros(
        (6, GRID_SIZE, GRID_SIZE),
        dtype=np.float32,
    )

    for atom_index, atom in enumerate(mol.GetAtoms()):
        x, y = coordinates[atom_index]

        distance = np.sqrt(
            (GRID_X - x) ** 2
            + (GRID_Y - y) ** 2
        ) + 0.25

        atomic_number = atom.GetAtomicNum()
        vdw_radius = float(
            PERIODIC_TABLE.GetRvdw(atomic_number) or 1.7
        )

        vdw_field = 1.0 - np.exp(
            -np.clip((vdw_radius / distance) ** 12, 0, 50)
        )

        if atom_index in donor_ids or atom_index in acceptor_ids:
            c_value, d_value = (
                (1.0, 1.2)
                if atomic_number in (7, 8)
                else (0.8, 1.0)
            )

            h_bond_field = np.abs(
                c_value / np.clip(distance, 0.65, None) ** 12
                - d_value / np.clip(distance, 0.65, None) ** 10
            )

            channels[0] += np.clip(h_bond_field, 0, 10)

        if atomic_number in HYDROPHOBIC_ATOMS:
            channels[1] += vdw_field

        if atomic_number in METALS:
            channels[2] += vdw_field

        channels[3] += vdw_field

        if (
            atom.GetFormalCharge() > 0
            or (
                atomic_number in (7, 15)
                and atom.GetTotalDegree() >= 4
            )
        ):
            channels[4] += vdw_field

        if (
            atom.GetFormalCharge() < 0
            or (
                atomic_number in (8, 16)
                and atom.GetTotalDegree() == 1
            )
        ):
            channels[5] += vdw_field

    for channel_index in range(6):
        maximum = float(
            np.percentile(channels[channel_index], 99.5)
        )

        if maximum > 0:
            channels[channel_index] = np.clip(
                channels[channel_index] / maximum,
                0,
                1,
            )

    return channels.astype(np.float16)

In [ ]:
grid_path = FEATURE_DIR / "six_channel_grids_float16.npy"

X_grids = np.lib.format.open_memmap(
    grid_path,
    mode="w+",
    dtype="float16",
    shape=(len(df), 6, GRID_SIZE, GRID_SIZE),
)

for i, mol in enumerate(tqdm(
    df["mol"],
    desc="Generating six-channel molecular grids",
)):
    X_grids[i] = molecule_six_channel_grid(mol)

X_grids.flush()
X_grids = np.load(grid_path, mmap_mode="r")

print("Grid tensor shape:", X_grids.shape)

In [ ]:
channel_names = [
    "Hydrogen Bond",
    "Hydrophobicity",
    "Metallicity",
    "Excluded Volume",
    "Positive Ionization",
    "Negative Ionization",
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
sample_grid = np.asarray(X_grids[0], dtype=np.float32)

for ax, grid, name in zip(axes.flat, sample_grid, channel_names):
    image = ax.imshow(grid, cmap="magma", origin="lower")
    ax.set_title(name, fontweight="bold")
    ax.axis("off")
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    "Multi-channel 2D Molecular Grid",
    fontweight="bold",
    fontsize=15,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "03_multichannel_grid_sample.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

In [ ]:
def translate_grid_with_zeros(grid, shift_y, shift_x):
    translated = np.zeros_like(grid)

    source_y_start = max(0, -shift_y)
    source_y_end = min(grid.shape[1], grid.shape[1] - shift_y)
    source_x_start = max(0, -shift_x)
    source_x_end = min(grid.shape[2], grid.shape[2] - shift_x)

    target_y_start = max(0, shift_y)
    target_y_end = target_y_start + (source_y_end - source_y_start)
    target_x_start = max(0, shift_x)
    target_x_end = target_x_start + (source_x_end - source_x_start)

    translated[
        :,
        target_y_start:target_y_end,
        target_x_start:target_x_end,
    ] = grid[
        :,
        source_y_start:source_y_end,
        source_x_start:source_x_end,
    ]

    return translated

In [ ]:
class GridDataset(Dataset):
    def __init__(self, grid_array, indices, labels, augment=False):
        self.grid_array = grid_array
        self.indices = np.asarray(indices, dtype=int)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        grid = np.asarray(
            self.grid_array[self.indices[index]],
            dtype=np.float32,
        ).copy()

        if self.augment:
            rotation = np.random.randint(0, 4)
            grid = np.rot90(
                grid,
                k=rotation,
                axes=(1, 2),
            ).copy()

            if np.random.rand() < 0.5:
                grid = grid[:, :, ::-1].copy()

            shift_y = np.random.randint(-2, 3)
            shift_x = np.random.randint(-2, 3)
            grid = translate_grid_with_zeros(
                grid,
                shift_y,
                shift_x,
            )

            channel_scale = np.random.uniform(
                0.92,
                1.08,
                size=(6, 1, 1),
            ).astype(np.float32)

            grid *= channel_scale

            if np.random.rand() < 0.40:
                grid += np.random.normal(
                    0.0,
                    0.008,
                    size=grid.shape,
                ).astype(np.float32)

            grid = np.clip(grid, 0.0, 1.0)

        return (
            torch.from_numpy(grid),
            torch.from_numpy(self.labels[index]),
        )

In [ ]:
def norm2d(channels):
    groups = 8 if channels % 8 == 0 else 4
    return nn.GroupNorm(groups, channels)


class MultiChannel2DCNN(nn.Module):
    def __init__(self, number_of_tasks):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(6, 24, 3, padding=1, bias=False),
            norm2d(24),
            nn.SiLU(),
            nn.Dropout2d(0.08),
            nn.MaxPool2d(2),

            nn.Conv2d(24, 48, 3, padding=1, bias=False),
            norm2d(48),
            nn.SiLU(),
            nn.Dropout2d(0.10),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 96, 3, padding=1, bias=False),
            norm2d(96),
            nn.SiLU(),
            nn.Dropout2d(0.12),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, 3, padding=1, bias=False),
            norm2d(128),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.45),
            nn.Linear(128, 96),
            nn.SiLU(),
            nn.Dropout(0.35),
            nn.Linear(96, number_of_tasks),
        )

    def forward(self, X):
        return self.head(self.features(X))

In [ ]:
@torch.no_grad()
def predict_grid_tta(
    model,
    indices,
    labels,
    rotations=(0, 1, 2, 3),
):
    dataset = GridDataset(
        X_grids,
        indices,
        labels,
        augment=False,
    )

    loader = DataLoader(
        dataset,
        batch_size=max(BATCH_IMAGE, 24),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    model.eval()
    predictions = []

    for X_batch, _ in loader:
        X_batch = X_batch.to(DEVICE)
        rotation_predictions = []

        for rotation in rotations:
            rotated = torch.rot90(
                X_batch,
                k=rotation,
                dims=(2, 3),
            )

            rotation_predictions.append(
                torch.sigmoid(model(rotated))
            )

        predictions.append(
            torch.stack(
                rotation_predictions,
                dim=0,
            ).mean(dim=0).cpu().numpy()
        )

    return np.vstack(predictions)

In [ ]:
def build_cnn_sampler():
    endpoint_positive = np.nansum(Y_train == 1, axis=0)
    endpoint_negative = np.nansum(Y_train == 0, axis=0)
    rarity = endpoint_negative / np.maximum(endpoint_positive, 1)

    sample_weights = np.ones(len(train_idx), dtype=np.float64)

    for i in range(len(train_idx)):
        positive_tasks = np.where(Y_train[i] == 1)[0]

        if len(positive_tasks):
            moderate_weight = np.sqrt(
                np.minimum(rarity[positive_tasks], 16.0)
            )

            sample_weights[i] += min(
                float(np.mean(moderate_weight)),
                4.0,
            )

    return WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

In [ ]:
def train_grid_cnn():
    model_name = "Regularized Multi-channel 2D CNN"

    train_dataset = GridDataset(
        X_grids,
        train_idx,
        Y_train,
        augment=True,
    )

    validation_dataset = GridDataset(
        X_grids,
        val_idx,
        Y_val,
        augment=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_IMAGE,
        sampler=build_cnn_sampler(),
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=max(BATCH_IMAGE, 24),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    model = MultiChannel2DCNN(len(TARGETS)).to(DEVICE)

    pos_weight = compute_pos_weight(
        Y_train,
        cap=6.0,
        power=0.50,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=5e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": [],
        "learning_rate": [],
    }

    best_auc = -np.inf
    wait = 0
    top_states = []
    max_epochs = 120
    patience = 20
    top_k = 3

    for epoch in range(max_epochs):
        model.train()
        train_losses = []

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            Y_batch = Y_batch.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(X_batch)

            loss = masked_focal_bce(
                logits,
                Y_batch,
                pos_weight=pos_weight,
                gamma=1.25,
                label_smoothing=0.01,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=3.0,
            )
            optimizer.step()

            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        validation_probabilities = []
        validation_targets = []

        with torch.no_grad():
            for X_batch, Y_batch in validation_loader:
                X_batch = X_batch.to(DEVICE)
                Y_batch = Y_batch.to(DEVICE)

                logits = model(X_batch)

                validation_losses.append(float(
                    masked_focal_bce(
                        logits,
                        Y_batch,
                        pos_weight=pos_weight,
                        gamma=1.25,
                        label_smoothing=0.01,
                    ).cpu()
                ))

                validation_probabilities.append(
                    torch.sigmoid(logits).cpu().numpy()
                )

                validation_targets.append(
                    Y_batch.cpu().numpy()
                )

        train_loss = float(np.mean(train_losses))
        validation_loss = float(np.mean(validation_losses))
        validation_probability = np.vstack(validation_probabilities)
        validation_target = np.vstack(validation_targets)

        validation_auc = mean_metric([
            safe_auc(
                validation_target[:, j],
                validation_probability[:, j],
            )
            for j in range(len(TARGETS))
        ])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation_loss)
        history["val_auc"].append(validation_auc)
        history["learning_rate"].append(
            float(optimizer.param_groups[0]["lr"])
        )

        scheduler.step(validation_auc)

        top_states.append({
            "epoch": epoch + 1,
            "val_auc": float(validation_auc),
            "state_dict": clone_state_dict(model),
        })

        top_states = sorted(
            top_states,
            key=lambda item: item["val_auc"],
            reverse=True,
        )[:top_k]

        if validation_auc > best_auc + 2e-4:
            best_auc = validation_auc
            wait = 0
        else:
            wait += 1

        print(
            f"Epoch {epoch + 1:03d}/{max_epochs} | "
            f"train_loss={train_loss:.5f} | "
            f"val_loss={validation_loss:.5f} | "
            f"val_auc={validation_auc:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if epoch + 1 >= 20 and wait >= patience:
            print("Validation-AUC early stopping activated.")
            break

    averaged_state = average_state_dicts([
        item["state_dict"]
        for item in top_states
    ])

    model.load_state_dict(averaged_state)
    model = model.to(DEVICE)

    print(
        "CNN top checkpoint epochs:",
        [
            (item["epoch"], round(item["val_auc"], 4))
            for item in top_states
        ],
    )

    P_train = predict_grid_tta(
        model,
        train_idx,
        Y_train,
        rotations=(0, 1),
    )

    P_validation = predict_grid_tta(
        model,
        val_idx,
        Y_val,
        rotations=(0, 1, 2, 3),
    )

    P_test = predict_grid_tta(
        model,
        test_idx,
        Y_test,
        rotations=(0, 1, 2, 3),
    )

    history["best_val_auc"] = float(best_auc)
    history["top_checkpoint_records"] = [
        {
            "epoch": item["epoch"],
            "val_auc": item["val_auc"],
        }
        for item in top_states
    ]

    torch.save(
        model.state_dict(),
        MODEL_DIR / "multichannel_2d_cnn.pt",
    )

    return register_model_result(
        model_name,
        P_train,
        P_validation,
        P_test,
        history=history,
        diagnostic_method=(
            "Regularized train prediction with two-rotation TTA; "
            "validation-AUC early stopping and checkpoint averaging"
        ),
        notes=(
            "Six-channel 24A molecular-grid CNN with zero-padded translation, "
            "GroupNorm, spatial dropout, moderate sampling, focal BCE, "
            "geometric augmentation, validation-AUC early stopping, "
            "checkpoint averaging and four-rotation validation/test TTA."
        ),
    )

In [ ]:
cnn_summary, cnn_per_endpoint = train_grid_cnn()

## Model 9 — Strongly Regularized Compact MTL-DNN

The previous `1024-512-128` network overfit. The revised architecture is `512-256-64`, with stronger dropout, stronger weight decay, stricter rare-bit filtering and mild focal BCE.

In [ ]:
MTL_DESCRIPTOR_FUNCTIONS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Crippen.MolLogP),
    ("HBD", Lipinski.NumHDonors),
    ("HBA", Lipinski.NumHAcceptors),
    ("TPSA", rdMolDescriptors.CalcTPSA),
    ("RotB", Lipinski.NumRotatableBonds),
    ("RingCount", rdMolDescriptors.CalcNumRings),
]

X_mtl_descriptors = np.asarray([
    [
        float(function(mol))
        for _, function in MTL_DESCRIPTOR_FUNCTIONS
    ]
    for mol in tqdm(df["mol"], desc="Compact MTL descriptors")
], dtype=np.float32)

In [ ]:
mtl_imputer = SimpleImputer(strategy="median")
mtl_scaler = StandardScaler()

D_train = mtl_scaler.fit_transform(
    mtl_imputer.fit_transform(X_mtl_descriptors[train_idx])
)
D_val = mtl_scaler.transform(
    mtl_imputer.transform(X_mtl_descriptors[val_idx])
)
D_test = mtl_scaler.transform(
    mtl_imputer.transform(X_mtl_descriptors[test_idx])
)

M_train = X_morgan[train_idx].astype(np.float32)
M_val = X_morgan[val_idx].astype(np.float32)
M_test = X_morgan[test_idx].astype(np.float32)

bit_count = np.sum(M_train > 0.5, axis=0)
MTL_KEEP_MASK = (
    (bit_count >= 10)
    & (bit_count <= int(len(M_train) * 0.995))
)

X_train_mtl = np.hstack([
    M_train[:, MTL_KEEP_MASK],
    D_train,
]).astype(np.float32)

X_val_mtl = np.hstack([
    M_val[:, MTL_KEEP_MASK],
    D_val,
]).astype(np.float32)

X_test_mtl = np.hstack([
    M_test[:, MTL_KEEP_MASK],
    D_test,
]).astype(np.float32)

print(
    f"Morgan bits retained: {int(MTL_KEEP_MASK.sum())} / {N_BITS}"
)
print("Compact MTL input shape:", X_train_mtl.shape)

In [ ]:
joblib.dump({
    "imputer": mtl_imputer,
    "scaler": mtl_scaler,
    "descriptor_names": [
        item[0]
        for item in MTL_DESCRIPTOR_FUNCTIONS
    ],
    "morgan_keep_mask": MTL_KEEP_MASK,
}, MODEL_DIR / "compact_mtl_preprocessing.joblib")

In [ ]:
class CompactMTLDNN(nn.Module):
    def __init__(self, input_dimension, number_of_tasks):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.50),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.45),

            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.35),

            nn.Linear(64, number_of_tasks),
        )

    def forward(self, X):
        return self.network(X)

In [ ]:
mtl_model = CompactMTLDNN(
    X_train_mtl.shape[1],
    len(TARGETS),
).to(DEVICE)

mtl_pos_weight = compute_pos_weight(
    Y_train,
    cap=8.0,
    power=0.65,
)

mtl_optimizer = torch.optim.AdamW(
    mtl_model.parameters(),
    lr=1.8e-4,
    weight_decay=7e-4,
)

mtl_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    mtl_optimizer,
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

In [ ]:
mtl_summary, mtl_per_endpoint = train_tabular_multitask(
    "Strongly Regularized Compact MTL-DNN",
    mtl_model,
    X_train_mtl,
    Y_train,
    X_val_mtl,
    Y_val,
    X_test_mtl,
    optimizer=mtl_optimizer,
    max_epochs=180,
    batch_size=BATCH_TABULAR,
    patience=20,
    minimum_epochs=15,
    loss_function=lambda raw, y: masked_focal_bce(
        raw,
        y,
        pos_weight=mtl_pos_weight,
        gamma=0.75,
        label_smoothing=0.02,
    ),
    scheduler=mtl_scheduler,
    top_k=3,
    notes=(
        "Reduced 512-256-64 MTL-DNN with stricter ECFP4 rare-bit filtering, "
        "seven descriptors, LayerNorm, stronger dropout, stronger AdamW "
        "weight decay, masked focal BCE, validation-AUC early stopping and "
        "in-memory checkpoint averaging."
    ),
)

## Model 10 — Validation-Selected Rank Ensemble

This is not hard voting. It combines continuous ranked prediction scores. For every endpoint, weak models are filtered using validation ROC-AUC, strong candidates are greedily added only when validation ROC-AUC improves, and repeated selections become weights.

The previous same-validation logistic calibration remains removed. Final Accuracy is calculated from the ensemble score with the same fixed 0.50 threshold used by every other model.


In [ ]:
def rank01(values):
    values = np.asarray(values, dtype=float)
    output = np.full(len(values), 0.5, dtype=float)
    mask = np.isfinite(values)

    if mask.sum() <= 1:
        return output

    ranks = rankdata(values[mask], method="average")
    output[mask] = (
        (ranks - 1.0)
        / max(len(ranks) - 1.0, 1.0)
    )

    return output

In [ ]:
base_model_names = [
    name
    for name in MODEL_PREDICTIONS
    if name != "Validation-Selected Rank Ensemble"
]

if len(base_model_names) < 2:
    raise RuntimeError(
        "At least two completed base models are required for the ensemble."
    )

print("Base models available for the ensemble:")
for name in base_model_names:
    print("-", name)

In [ ]:
number_of_models = len(base_model_names)
number_of_tasks = len(TARGETS)

ensemble_weights = np.zeros(
    (number_of_models, number_of_tasks),
    dtype=float,
)

P_train_ensemble = np.zeros(
    (len(train_idx), number_of_tasks),
    dtype=float,
)

P_validation_ensemble = np.zeros(
    (len(val_idx), number_of_tasks),
    dtype=float,
)

P_test_ensemble = np.zeros(
    (len(test_idx), number_of_tasks),
    dtype=float,
)

selection_rows = []

In [ ]:
for j, target in enumerate(TARGETS):
    validation_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name]["validation"][:, j]
        )
        for name in base_model_names
    ])

    train_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name]["diagnostic_train"][:, j]
        )
        for name in base_model_names
    ])

    test_ranks = np.vstack([
        rank01(
            MODEL_PREDICTIONS[name]["test"][:, j]
        )
        for name in base_model_names
    ])

    validation_aucs = np.asarray([
        safe_auc(
            Y_val[:, j],
            validation_ranks[model_index],
        )
        for model_index in range(number_of_models)
    ], dtype=float)

    finite_models = np.where(np.isfinite(validation_aucs))[0]

    if len(finite_models) == 0:
        selected_models = [0]
    else:
        best_model_index = finite_models[
            np.argmax(validation_aucs[finite_models])
        ]

        best_auc = validation_aucs[best_model_index]

        eligible_models = [
            model_index
            for model_index in finite_models
            if validation_aucs[model_index]
            >= max(0.65, best_auc - 0.05)
        ]

        eligible_models = sorted(
            eligible_models,
            key=lambda index: validation_aucs[index],
            reverse=True,
        )[:5]

        selected_models = [best_model_index]
        current_score = validation_ranks[best_model_index].copy()
        current_auc = safe_auc(Y_val[:, j], current_score)

        for _ in range(1, 6):
            best_candidate = None
            best_candidate_auc = current_auc
            best_candidate_score = None

            for model_index in eligible_models:
                trial_score = (
                    current_score * len(selected_models)
                    + validation_ranks[model_index]
                ) / (len(selected_models) + 1)

                trial_auc = safe_auc(
                    Y_val[:, j],
                    trial_score,
                )

                if (
                    np.isfinite(trial_auc)
                    and trial_auc > best_candidate_auc + 0.001
                ):
                    best_candidate = model_index
                    best_candidate_auc = trial_auc
                    best_candidate_score = trial_score

            if best_candidate is None:
                break

            selected_models.append(best_candidate)
            current_score = best_candidate_score
            current_auc = best_candidate_auc

    selection_counts = Counter(selected_models)

    for model_index, count in selection_counts.items():
        ensemble_weights[model_index, j] = (
            count / len(selected_models)
        )

    P_train_ensemble[:, j] = np.average(
        train_ranks,
        axis=0,
        weights=ensemble_weights[:, j],
    )

    P_validation_ensemble[:, j] = np.average(
        validation_ranks,
        axis=0,
        weights=ensemble_weights[:, j],
    )

    P_test_ensemble[:, j] = np.average(
        test_ranks,
        axis=0,
        weights=ensemble_weights[:, j],
    )

    selection_rows.append({
        "Endpoint": target,
        "Selected Models": " | ".join(
            f"{base_model_names[index]} x{selection_counts[index]}"
            for index in selection_counts
        ),
        "Validation AUC Before Blend": float(
            np.nanmax(validation_aucs)
        ) if np.isfinite(validation_aucs).any() else np.nan,
        "Validation AUC After Blend": safe_auc(
            Y_val[:, j],
            P_validation_ensemble[:, j],
        ),
    })

In [ ]:
ensemble_weight_table = pd.DataFrame(
    ensemble_weights,
    index=base_model_names,
    columns=TARGETS,
)

ensemble_selection_table = pd.DataFrame(selection_rows)

ensemble_weight_table.to_csv(
    OUTPUT_DIR / "validation_selected_rank_weights.csv"
)
ensemble_selection_table.to_csv(
    OUTPUT_DIR / "endpoint_selection_report.csv",
    index=False,
)


display(
    ensemble_weight_table.style
    .format("{:.3f}")
    .background_gradient(cmap="Blues")
    .set_caption(
        "Endpoint-wise Validation-Selected Rank Weights"
    )
)


display(
    ensemble_selection_table.style
    .format({
        "Validation AUC Before Blend": "{:.4f}",
        "Validation AUC After Blend": "{:.4f}",
    })
    .set_caption("Greedy Rank-Ensemble Selection Audit")
)

In [ ]:
ensemble_summary, ensemble_per_endpoint = register_model_result(
    "Validation-Selected Rank Ensemble",
    P_train_ensemble,
    P_validation_ensemble,
    P_test_ensemble,
    diagnostic_method=(
        "Weighted ranks of base diagnostic-train predictions; "
        "validation-only model selection"
    ),
    notes=(
        "Per-endpoint weak-model filtering and greedy weighted rank averaging. "
        "No test-label leakage and no same-validation logistic calibration."
    ),
)

# Final Results and Visualizations

In [ ]:
summary_df = pd.DataFrame(
    MODEL_RESULTS.values()
).sort_values(
    "Test Mean AUC",
    ascending=False,
).reset_index(drop=True)

endpoint_results_df = pd.concat([
    table.assign(Model=model_name)
    for model_name, table in MODEL_ENDPOINT_RESULTS.items()
], ignore_index=True)

summary_df.to_csv(
    OUTPUT_DIR / "final_model_leaderboard.csv",
    index=False,
)

endpoint_results_df.to_csv(
    OUTPUT_DIR / "final_endpoint_results.csv",
    index=False,
)

leaderboard_columns = [
    "Model",
    "Diagnostic Train Mean AUC",
    "Validation Mean AUC",
    "Test Mean AUC",
    "Test Mean Accuracy",
    "Train-Validation AUC Gap",
    "Fitting Status",
]

display(
    summary_df[leaderboard_columns]
    .style
    .format({
        "Diagnostic Train Mean AUC": "{:.4f}",
        "Validation Mean AUC": "{:.4f}",
        "Test Mean AUC": "{:.4f}",
        "Test Mean Accuracy": "{:.4f}",
        "Train-Validation AUC Gap": "{:+.4f}",
    })
    .background_gradient(
        subset=["Test Mean AUC"],
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
    )
    .set_caption(
        "Final Model Leaderboard — ROC-AUC and Accuracy"
    )
)


In [ ]:
if len(summary_df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    auc_table = summary_df.sort_values("Test Mean AUC")
    axes[0].barh(
        auc_table["Model"],
        auc_table["Test Mean AUC"],
    )
    axes[0].axvline(
        TARGET_MEAN_AUC,
        linestyle="--",
        label=f"Target = {TARGET_MEAN_AUC:.2f}",
    )
    axes[0].set_xlim(0.5, 1.02)
    axes[0].set_title(
        "Mean Test ROC-AUC",
        fontweight="bold",
    )
    axes[0].legend()

    accuracy_table = summary_df.sort_values(
        "Test Mean Accuracy"
    )
    axes[1].barh(
        accuracy_table["Model"],
        accuracy_table["Test Mean Accuracy"],
    )
    axes[1].set_xlim(0.5, 1.02)
    axes[1].set_title(
        "Mean Test Accuracy",
        fontweight="bold",
    )

    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "10_model_comparison_auc_accuracy.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
if len(summary_df):
    plt.figure(figsize=(12, 7))

    statuses = summary_df["Fitting Status"].unique().tolist()
    palette = dict(zip(
        statuses,
        sns.color_palette("Set2", n_colors=max(3, len(statuses))),
    ))

    for status, group in summary_df.groupby("Fitting Status"):
        plt.scatter(
            group["Diagnostic Train Mean AUC"],
            group["Validation Mean AUC"],
            s=120,
            label=status,
            color=palette[status],
            edgecolor="black",
            alpha=0.9,
        )

        for _, row in group.iterrows():
            plt.annotate(
                row["Model"],
                (
                    row["Diagnostic Train Mean AUC"],
                    row["Validation Mean AUC"],
                ),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

    lower_limit = min(
        0.5,
        summary_df[[
            "Diagnostic Train Mean AUC",
            "Validation Mean AUC",
        ]].min().min() - 0.02,
    )

    plt.plot(
        [lower_limit, 1],
        [lower_limit, 1],
        "--",
        label="Diagnostic Train = Validation",
    )

    plt.xlim(lower_limit, 1.01)
    plt.ylim(lower_limit, 1.01)
    plt.xlabel("Diagnostic Train Mean AUC")
    plt.ylabel("Validation Mean AUC")
    plt.title(
        "Fitting Diagnostic after Regularization",
        fontweight="bold",
    )
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "11_fitting_diagnostic.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
if len(endpoint_results_df):
    auc_pivot = endpoint_results_df.pivot(
        index="Endpoint",
        columns="Model",
        values="Test AUC",
    )

    plt.figure(
        figsize=(max(14, 1.25 * len(auc_pivot.columns)), 7)
    )

    sns.heatmap(
        auc_pivot,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
        linewidths=0.4,
        cbar_kws={"label": "Test ROC-AUC"},
    )

    plt.title(
        "Test ROC-AUC Heatmap — Models x Tox21 Endpoints",
        fontweight="bold",
    )
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "12_auc_heatmap.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
if len(endpoint_results_df):
    accuracy_pivot = endpoint_results_df.pivot(
        index="Endpoint",
        columns="Model",
        values="Test Accuracy",
    )

    plt.figure(
        figsize=(
            max(14, 1.25 * len(accuracy_pivot.columns)),
            7,
        )
    )

    sns.heatmap(
        accuracy_pivot,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        vmin=0.5,
        vmax=1.0,
        linewidths=0.4,
        cbar_kws={"label": "Test Accuracy"},
    )

    plt.title(
        "Test Accuracy Heatmap — Models x Tox21 Endpoints",
        fontweight="bold",
    )
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "13_accuracy_heatmap.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
if len(summary_df):
    best_model = summary_df.iloc[0]["Model"]
    best_rows = MODEL_ENDPOINT_RESULTS[best_model].copy()
    x_positions = np.arange(len(best_rows))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].bar(
        x_positions,
        best_rows["Test AUC"],
    )
    axes[0].axhline(
        best_rows["Test AUC"].mean(),
        linestyle="--",
        label=(
            f"Mean = "
            f"{best_rows['Test AUC'].mean():.3f}"
        ),
    )
    axes[0].set_ylim(0.5, 1.03)
    axes[0].set_title(
        f"ROC-AUC by Endpoint — {best_model}",
        fontweight="bold",
    )

    axes[1].bar(
        x_positions,
        best_rows["Test Accuracy"],
    )
    axes[1].axhline(
        best_rows["Test Accuracy"].mean(),
        linestyle="--",
        label=(
            f"Mean = "
            f"{best_rows['Test Accuracy'].mean():.3f}"
        ),
    )
    axes[1].set_ylim(0.5, 1.03)
    axes[1].set_title(
        f"Accuracy by Endpoint — {best_model}",
        fontweight="bold",
    )

    for ax in axes:
        ax.set_xticks(x_positions)
        ax.set_xticklabels(
            best_rows["Endpoint"],
            rotation=45,
            ha="right",
        )
        ax.legend()

    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "14_best_model_endpoint_auc_accuracy.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

    print("Best model by Mean Test AUC:", best_model)


In [ ]:
# Confusion-matrix evaluation was intentionally removed.
# This notebook reports only ROC-AUC and Accuracy.


In [ ]:
if MODEL_HISTORIES:
    number_of_histories = len(MODEL_HISTORIES)

    fig, axes = plt.subplots(
        number_of_histories,
        1,
        figsize=(12, max(4, 4.0 * number_of_histories)),
        squeeze=False,
    )

    for ax, (model_name, history) in zip(
        axes.flat,
        MODEL_HISTORIES.items(),
    ):
        ax.plot(
            history["train_loss"],
            label="Train Loss",
            linewidth=2,
        )
        ax.plot(
            history["val_loss"],
            label="Validation Loss",
            linewidth=2,
        )

        ax.set_title(model_name, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.25)

        if history.get("val_auc"):
            second_axis = ax.twinx()
            second_axis.plot(
                history["val_auc"],
                linestyle="--",
                linewidth=1.8,
                alpha=0.75,
                label="Validation AUC",
            )
            second_axis.set_ylabel("Validation AUC")
            second_axis.set_ylim(0.45, 1.02)

            best_epoch = int(np.nanargmax(
                np.asarray(history["val_auc"], dtype=float)
            ))

            ax.axvline(
                best_epoch,
                linestyle=":",
                alpha=0.7,
                label=f"Best AUC epoch = {best_epoch + 1}",
            )

            lines_1, labels_1 = ax.get_legend_handles_labels()
            lines_2, labels_2 = second_axis.get_legend_handles_labels()

            ax.legend(
                lines_1 + lines_2,
                labels_1 + labels_2,
                loc="best",
            )
        else:
            ax.legend(loc="best")

    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "16_neural_training_curves.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("No neural training history is available.")

In [ ]:
if CV_TABLES:
    all_cv_rows = pd.concat(
        CV_TABLES.values(),
        ignore_index=True,
    )

    cv_summary = (
        all_cv_rows
        .groupby("Model")
        .agg(
            CV_Mean_AUC=("CV Mean AUC", "mean"),
            CV_Std_AUC=("CV Mean AUC", "std"),
            CV_Mean_Accuracy=("CV Mean Accuracy", "mean"),
        )
        .reset_index()
        .sort_values("CV_Mean_AUC", ascending=False)
    )

    cv_summary.to_csv(
        OUTPUT_DIR / "three_fold_cv_summary.csv",
        index=False,
    )

    display(
        cv_summary.style
        .format({
            "CV_Mean_AUC": "{:.4f}",
            "CV_Std_AUC": "{:.4f}",
            "CV_Mean_Accuracy": "{:.4f}",
        })
        .set_caption("Three-Fold Cross-Validation Summary")
    )

    plt.figure(figsize=(12, 5))
    x_positions = np.arange(len(cv_summary))

    plt.bar(
        x_positions,
        cv_summary["CV_Mean_AUC"],
        yerr=cv_summary["CV_Std_AUC"].fillna(0),
        capsize=5,
    )

    plt.xticks(
        x_positions,
        cv_summary["Model"],
        rotation=25,
        ha="right",
    )
    plt.ylim(0.5, 1.02)
    plt.ylabel("Mean CV AUC")
    plt.title(
        "Three-Fold Cross-Validation — Mean AUC",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "17_three_fold_cv.png",
        dpi=240,
        bbox_inches="tight",
    )
    plt.show()

## Final export and reproducibility metadata

This section exports the final ROC-AUC/Accuracy leaderboard, endpoint-level ROC-AUC/Accuracy results, model predictions, figures and environment metadata. The notebook does not claim that the target AUC was reached unless the newly executed results show it.


In [ ]:
import sklearn
import rdkit
import xgboost

package_versions = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "rdkit": rdkit.__version__,
    "xgboost": xgboost.__version__,
    "torch": torch.__version__,
    "torchvision_available": TORCHVISION_AVAILABLE,
    "device": str(DEVICE),
    "gpu": (
        torch.cuda.get_device_name(0)
        if DEVICE.type == "cuda"
        else None
    ),
    "ram_gb": round(RAM_GB, 2),
    "resource_mode": RESOURCE_MODE,
}

run_configuration = {
    "notebook": "Updated_Shohoj_Pro_Max_Ultra.ipynb",
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": RAW_SHA256,
    "cleaned_samples": len(df),
    "split": {
        "train": TRAIN_RATIO,
        "validation": VALID_RATIO,
        "test": TEST_RATIO,
    },
    "seed": SEED,
    "metrics": [
        "Mean ROC-AUC",
        "Mean Accuracy",
    ],
    "classification_threshold": "Fixed probability threshold = 0.50",
    "pipeline_version": PIPELINE_VERSION,
    "fitting_diagnostic": (
        "Classical models use 3-fold OOF train predictions. Neural models "
        "use best-checkpoint train predictions selected by validation AUC."
    ),
    "target_mean_auc": TARGET_MEAN_AUC,
    "missing_label_strategy": (
        "Mask and ignore missing targets. Never impute target labels as 0 or 1."
    ),
    "persistent_kernel_restart_recovery": True,
    "hybrid_model": (
        "DenseNet121 visual PCA features + chemical features -> regularized XGBoost"
    ),
    "packages": package_versions,
    "completed_models": summary_df["Model"].tolist(),
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}

(OUTPUT_DIR / "run_config_and_environment.json").write_text(
    json.dumps(run_configuration, indent=2),
    encoding="utf-8",
)

endpoint_summary_raw.to_csv(
    OUTPUT_DIR / "endpoint_missing_imbalance_summary.csv",
    index=False,
)

split_distribution.to_csv(
    OUTPUT_DIR / "split_endpoint_distribution.csv",
    index=False,
)

In [ ]:
print("=" * 92)
print("Updated_Shohoj_Pro_Max_Ultra pipeline export complete.")
print(
    "Final leaderboard:",
    OUTPUT_DIR / "final_model_leaderboard.csv",
)
print(
    "Endpoint results:",
    OUTPUT_DIR / "final_endpoint_results.csv",
)
print("Figures:", FIGURE_DIR)
print("Models:", MODEL_DIR)
print("=" * 92)

if len(summary_df):
    best_result = summary_df.iloc[0]

    print("Best model:", best_result["Model"])
    print(
        f"Mean Test AUC: "
        f"{best_result['Test Mean AUC']:.4f}"
    )
    print(
        f"Mean Test Accuracy: "
        f"{best_result['Test Mean Accuracy']:.4f}"
    )

    if best_result["Test Mean AUC"] >= TARGET_MEAN_AUC:
        print(
            "The target Mean AUC of 0.90 was achieved "
            "in this run."
        )
    else:
        print(
            "The target Mean AUC of 0.90 was not achieved "
            "in this run. The reported result is the "
            "measured result."
        )


## Research implementation notes

- The evaluation output is intentionally restricted to ROC-AUC and Accuracy.
- Accuracy uses a fixed threshold of 0.50 for every endpoint. This is expected to increase ordinary Accuracy relative to the previous low validation-selected thresholds, especially on imbalanced endpoints. It does not alter ROC-AUC.
- The strongest methodological safeguards remain train-only preprocessing, missing-label masking, 3-fold OOF diagnostics for classical models, validation-AUC early stopping and no test-label use during model selection.
- The hybrid branch remains a boosted DenseNet121 visual-feature + chemical-feature XGBoost model.
- Reduced DNN capacity and stronger weight decay are intended to reduce the train-validation gap, but only a complete rerun can confirm the actual effect.
- The six-channel grid remains an RDKit-based approximation and should not be described as an exact reproduction of the proprietary ChemAxon/AutoDock workflow.
- The DenseNet visual branch uses ImageNet-pretrained features, not a chemistry-specific pretrained image encoder.
- The first cell restores the namespace through the last successfully completed cell. An interrupted cell restarts from the beginning of that cell.


## References

1. Mayr, A., Klambauer, G., Unterthiner, T., & Hochreiter, S. (2016). *DeepTox: Toxicity Prediction using Deep Learning*. Frontiers in Environmental Science, 3, 80.
2. Wang, Y., et al. (2021). *Multitask CapsNet: An Imbalanced Data Deep Learning Method for Predicting Toxicants*. ACS Omega, 6, 26545–26555.
3. Yuan, Q., et al. (2019). *Toxicity Prediction Method Based on Multi-Channel Convolutional Neural Network*. Molecules, 24, 3383.
4. User-supplied MTL-DNN paper for the compact multi-task architecture.
5. `Updated_shohoj_Pro.ipynb` and `Updated_shohoj_Pro.pdf`, reviewed as the implementation and result-audit sources for this revision.